# 🏛️ ĐỀ TÀI NGHIÊN CỨU KHOA HỌC: PhoBERT + GNN (RGCN) + SLIDING WINDOW
## Trích xuất thực thể và quan hệ từ văn bản pháp luật

Notebook **chạy độc lập**, giữ cách trình bày 11 bước của bản cũ. Chỉ cần file `.ipynb`
và dataset trên Google Drive; không cần upload hay chạy các file `.py` bên ngoài.

PhoBERT-base-v2 được đóng băng; huấn luyện NER start/end và RE/RGCN. Không dùng quan hệ
đáp án làm cạnh GNN. Train/validation/test khoảng 80/10/10 theo văn bản. Kết quả chính:
NER Precision/Recall/F1; RE micro-Precision/Recall/F1 và Macro-F1 trên 22 nhãn.
Cuối notebook: tải lên một DOCX để nhận diện, xuất CSV/JSON và đồ thị HTML tương tác.


### 🚀 [BƯỚC 1] Kết nối Google Drive & Kiểm tra môi trường
Bật **Runtime → Change runtime type → GPU** trước khi chạy. Chạy lần lượt từ trên xuống.


In [ ]:
import sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    print('✅ Google Drive đã kết nối.')
else:
    print('💻 Môi trường local: sửa DATASET_DIR tại Bước 3 theo máy của bạn.')


### 📦 [BƯỚC 2] Cài đặt Thư viện Phụ thuộc
Chỉ cài thư viện công khai; mã xử lý dữ liệu và mô hình đã nằm ngay trong notebook.


In [ ]:
get_ipython().run_line_magic('pip', "install -q 'torch>=2.6,<3' 'torch_geometric>=2.5,<3' 'transformers>=4.40,<6' 'pyvi==0.1.1' 'scikit-learn>=1.4,<2' 'matplotlib>=3.8,<4' 'pandas>=2.1,<4' 'numpy>=1.26,<3' 'networkx>=3.2,<4' tabulate")
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('✅ Thư viện đã sẵn sàng.')
print('⚙️ Thiết bị:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU — train toàn bộ sẽ chậm')


### 📁 [BƯỚC 3] Thiết lập Đường dẫn Dataset trên Google Drive
Chỉ sửa `DATASET_DIR` và `OUTPUT_DIR`. Tên thư mục phân biệt chữ hoa/thường.
Không cần `PROJECT_DIR`. Dataset phải đã giải nén và có các file `*_ket_qua.json`.


In [ ]:
from pathlib import Path

TRAIN_NEW_MODEL = True  # False: chỉ nạp checkpoint đã train để kiểm tra DOCX
TRAIN_SEED = 42

DATASET_DIR = Path('/content/drive/MyDrive/NCKH_train/Dataset')
OUTPUT_DIR = Path('/content/drive/MyDrive/NCKH_Results/phobert_rgcn_v2_run_42')
# Nếu chạy local, đổi 2 đường dẫn phía trên theo máy của bạn.

def check_dataset_path(path):
    path = Path(path)
    if not path.is_dir():
        raise FileNotFoundError(f'Không tìm thấy dataset: {path}. Kiểm tra kết nối Drive và tên thư mục ở Bước 3.')
    files = sorted(path.rglob('*_ket_qua.json'))
    if not files:
        names = ', '.join(p.name for p in list(path.iterdir())[:15]) or '(thư mục trống)'
        raise FileNotFoundError(f'Không có *_ket_qua.json trong {path}. Hãy giải nén dữ liệu nếu còn dạng ZIP. Nội dung thư mục: {names}')
    return files

if TRAIN_NEW_MODEL and (OUTPUT_DIR/'run_config.json').exists():
    raise ValueError('OUTPUT_DIR already contains a run. Use a new directory, or set TRAIN_NEW_MODEL=False to test DOCX.')
dataset_files = check_dataset_path(DATASET_DIR) if TRAIN_NEW_MODEL else []
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'✅ Dataset: {DATASET_DIR} — {len(dataset_files)} file JSON')
print(f'📁 Kết quả: {OUTPUT_DIR}')


### 🏷️ [BƯỚC 4] Hệ thống Nhãn & Bảng Ánh xạ Quan hệ
40 loại thực thể, 22 loại quan hệ dương. Một cặp có thể có nhiều quan hệ;
không có quan hệ khi không lớp nào vượt ngưỡng dự đoán.


In [ ]:
"""
labels.py
Định nghĩa hệ thống nhãn (Taxonomy & Label Schema) chuẩn hóa cho đề tài NCKH:
"Ứng dụng GNN kết hợp Transformer trong trích xuất quan hệ thực thể cấp độ tài liệu".

Đồng bộ 100% với:
  1. File cấu hình Label Studio: docs/label.xml
  2. Pipeline gán nhãn tự động: auto_label_pipeline.py
  3. 174 văn bản đã gán nhãn thực tế trong thư mục dataset/ (JSON & CSV)

Hệ thống bao gồm:
  - 40 Nhãn Thực thể (Entity Labels):
      + 12 trường metadata (TEN_VAN_BAN, SO_HIEU, CO_QUAN_BAN_HANH, CHUC_DANH, NGUOI_KY,...)
      + 7 cấp cấu trúc văn bản (PHAN, CHUONG, MUC, TIEU_MUC, DIEU, KHOAN, DIEM)
      + 21 loại văn bản viện dẫn / pháp lý liên quan (VB_HUONG_DAN_AP_DUNG, CAN_CU_BAN_HANH,...)
  - 22 Nhãn Quan hệ (Relation Labels):
      + 1 quan hệ cấu trúc & hành chính: THUOC (Thuộc)
      + 21 quan hệ pháp lý liên văn bản (CAN_CU_BAN_HANH, HUONG_DAN_AP_DUNG, QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH,...)
  - Hệ thống thuần túy 40 Thực thể & 22 Quan hệ chuẩn hóa (không sử dụng 10 cấp bậc đỉnh phân cấp).
"""
from typing import Dict, List, Optional, Set, Tuple
ENTITY_TAXONOMY: Dict[str, Tuple[int, str, str]] = {'TEN_VAN_BAN': (1, 'Tên văn bản', 'metadata'), 'SO_HIEU': (2, 'Số hiệu', 'metadata'), 'LOAI_VAN_BAN': (3, 'Loại văn bản', 'metadata'), 'NGANH': (4, 'Ngành', 'metadata'), 'NGAY_BAN_HANH': (5, 'Ngày ban hành', 'metadata'), 'LINH_VUC': (6, 'Lĩnh vực', 'metadata'), 'NGAY_CO_HIEU_LUC': (7, 'Ngày có hiệu lực', 'metadata'), 'NGAY_HET_HIEU_LUC': (8, 'Ngày hết hiệu lực', 'metadata'), 'TINH_TRANG_HIEU_LUC': (9, 'Tình trạng hiệu lực', 'metadata'), 'CO_QUAN_BAN_HANH': (10, 'Cơ quan ban hành', 'metadata'), 'CHUC_DANH': (11, 'Chức danh', 'metadata'), 'NGUOI_KY': (12, 'Người ký', 'metadata'), 'VB_DUOC_HUONG_DAN_AP_DUNG': (13, 'Văn bản được hướng dẫn áp dụng', 'reference'), 'VB_DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': (14, 'Văn bản được quy định chi tiết, hướng dẫn thi hành', 'reference'), 'VB_BI_TAC_DONG_HIEU_LUC_MOT_PHAN': (15, 'Văn bản bị tác động hiệu lực một phần', 'reference'), 'VB_BI_TAC_DONG_HIEU_LUC_TOAN_BO': (16, 'Văn bản bị tác động hiệu lực toàn bộ', 'reference'), 'VB_DUOC_DINH_CHINH': (17, 'Văn bản được đính chính', 'reference'), 'VB_DUOC_DAN_CHIEU': (18, 'Văn bản được dẫn chiếu', 'reference'), 'CAN_CU_BAN_HANH': (19, 'Căn cứ ban hành', 'reference'), 'VB_DUOC_GIAI_THICH': (20, 'Văn bản được giải thích', 'reference'), 'VB_BI_DINH_CHI_THI_HANH': (21, 'Văn bản bị đình chỉ thi hành', 'reference'), 'VB_BI_TAM_NGUNG_GIA_HAN': (22, 'Văn bản bị tạm ngưng/ gia hạn/ công bố tiếp tục còn hiệu lực', 'reference'), 'VB_HUONG_DAN_AP_DUNG': (23, 'Văn bản hướng dẫn áp dụng', 'reference'), 'VB_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': (24, 'Văn bản quy định chi tiết, hướng dẫn thi hành', 'reference'), 'VB_QUY_DINH_HET_HIEU_LUC_MOT_PHAN': (25, 'Văn bản quy định về việc hết hiệu lực một phần', 'reference'), 'VB_QUY_DINH_HET_HIEU_LUC_TOAN_BO': (26, 'Văn bản quy định về việc hết hiệu lực toàn bộ', 'reference'), 'VB_HOP_NHAT': (27, 'Văn bản hợp nhất', 'reference'), 'VB_DINH_CHINH': (28, 'Văn bản đính chính', 'reference'), 'VB_DAN_CHIEU': (29, 'Văn bản dẫn chiếu', 'reference'), 'VB_AP_DUNG': (30, 'Văn bản áp dụng', 'reference'), 'VB_GIAI_THICH': (31, 'Văn bản giải thích', 'reference'), 'VB_DINH_CHI_THI_HANH': (32, 'Văn bản đình chỉ thi hành', 'reference'), 'VB_QUY_DINH_TAM_NGUNG_GIA_HAN': (33, 'Văn bản quy định tạm ngưng/ gia hạn/ công bố tiếp tục còn hiệu lực', 'reference'), 'PHAN': (34, 'Phần', 'structure'), 'CHUONG': (35, 'Chương', 'structure'), 'MUC': (36, 'Mục', 'structure'), 'TIEU_MUC': (37, 'Tiểu Mục', 'structure'), 'DIEU': (38, 'Điều', 'structure'), 'KHOAN': (39, 'Khoản', 'structure'), 'DIEM': (40, 'Điểm', 'structure')}
THUOC_TINH_LABELS = [k for k, v in ENTITY_TAXONOMY.items() if v[2] == 'metadata']
DOC_REFERENCE_ENTITY_LABELS = [k for k, v in ENTITY_TAXONOMY.items() if v[2] == 'reference']
MUC_LUC_LABELS = [k for k, v in ENTITY_TAXONOMY.items() if v[2] == 'structure']
LUOC_DO_ENTITY_LABELS = DOC_REFERENCE_ENTITY_LABELS + ['van_ban_lien_quan']
ALL_ENTITIES = list(ENTITY_TAXONOMY.keys())
BIO_LABELS = ['o']
for entity in ALL_ENTITIES:
    BIO_LABELS.append(f'b-{entity.lower()}')
    BIO_LABELS.append(f'i-{entity.lower()}')
LABEL2ID = {label: idx for idx, label in enumerate(BIO_LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}
RELATION_TAXONOMY: Dict[str, Tuple[int, str]] = {'THUOC': (0, 'Thuộc'), 'CAN_CU_BAN_HANH': (1, 'Căn cứ ban hành'), 'HUONG_DAN_AP_DUNG': (2, 'Hướng dẫn áp dụng'), 'DUOC_HUONG_DAN_AP_DUNG': (3, 'Được hướng dẫn áp dụng'), 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': (4, 'Quy định chi tiết, hướng dẫn thi hành'), 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': (5, 'Được quy định chi tiết, hướng dẫn thi hành'), 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN': (6, 'Quy định về việc hết hiệu lực một phần'), 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN': (7, 'Bị tác động hiệu lực một phần'), 'QUY_DINH_HET_HIEU_LUC_TOAN_BO': (8, 'Quy định về việc hết hiệu lực toàn bộ'), 'BI_TAC_DONG_HIEU_LUC_TOAN_BO': (9, 'Bị tác động hiệu lực toàn bộ'), 'DINH_CHINH': (10, 'Đính chính'), 'DUOC_DINH_CHINH': (11, 'Được đính chính'), 'DAN_CHIEU': (12, 'Dẫn chiếu'), 'DUOC_DAN_CHIEU': (13, 'Được dẫn chiếu'), 'AP_DUNG': (14, 'Áp dụng'), 'GIAI_THICH': (15, 'Giải thích'), 'DUOC_GIAI_THICH': (16, 'Được giải thích'), 'HOP_NHAT': (17, 'Hợp nhất'), 'DINH_CHI_THI_HANH': (18, 'Đình chỉ thi hành'), 'BI_DINH_CHI_THI_HANH': (19, 'Bị đình chỉ thi hành'), 'QUY_DINH_TAM_NGUNG_GIA_HAN': (20, 'Quy định tạm ngưng/ gia hạn'), 'BI_TAM_NGUNG_GIA_HAN': (21, 'Bị tạm ngưng/ gia hạn')}
ALL_RELATIONS = list(RELATION_TAXONOMY.keys())
REL2ID = {r: idx for idx, r in enumerate(ALL_RELATIONS)}
ID2REL = {idx: r for idx, r in enumerate(ALL_RELATIONS)}
REL_VN_NAMES = {r: info[1] for r, info in RELATION_TAXONOMY.items()}
METADATA_RELATIONS: List[str] = []
STRUCTURAL_RELATIONS: List[str] = ['THUOC']
LEGAL_RELATIONS: List[str] = [r for r in ALL_RELATIONS if r != 'THUOC']
TEMPORAL_RELATIONS: List[str] = []
NON_TEMP_RELS = ALL_RELATIONS
LEVEL_LIST: List[str] = []
LEVEL2ID: Dict[str, int] = {}
ID2LEVEL: Dict[int, str] = {}
NUM_LEVELS: int = 0
ENTITY_TO_LEVEL_MAP: Dict[str, str] = {}

def get_node_level(code_or_label: str, scope: Optional[str]=None) -> str:
    """Tương thích ngược: trả về mã loại thực thể"""
    return str(code_or_label).strip() if code_or_label else 'entity'
RELATION_ALIAS_MAP: Dict[str, str] = {'THUOC': 'THUOC', 'THUỘC': 'THUOC', 'BELONGS_TO': 'THUOC', 'CONTAINS': 'THUOC', 'CAN_CU_BAN_HANH': 'CAN_CU_BAN_HANH', 'CĂN CỨ BAN HÀNH': 'CAN_CU_BAN_HANH', 'CĂN CỨ VÀO': 'CAN_CU_BAN_HANH', 'LÀM CĂN CỨ BAN HÀNH CHO': 'CAN_CU_BAN_HANH', 'DUOC_CAN_CU_BAN_HANH': 'CAN_CU_BAN_HANH', 'CAN_CU_VAO': 'CAN_CU_BAN_HANH', 'HUONG_DAN_AP_DUNG': 'HUONG_DAN_AP_DUNG', 'HƯỚNG DẪN ÁP DỤNG': 'HUONG_DAN_AP_DUNG', 'HUONG_DAN': 'HUONG_DAN_AP_DUNG', 'VB_HUONG_DAN_AP_DUNG': 'HUONG_DAN_AP_DUNG', 'DUOC_HUONG_DAN_AP_DUNG': 'DUOC_HUONG_DAN_AP_DUNG', 'ĐƯỢC HƯỚNG DẪN ÁP DỤNG': 'DUOC_HUONG_DAN_AP_DUNG', 'ĐƯỢC HƯỚNG DẪN ÁP DỤNG BỞI': 'DUOC_HUONG_DAN_AP_DUNG', 'DUOC_HUONG_DAN': 'DUOC_HUONG_DAN_AP_DUNG', 'VB_DUOC_HUONG_DAN_AP_DUNG': 'DUOC_HUONG_DAN_AP_DUNG', 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'CHI_TIET_HUONG_DAN_THI_HANH': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'QUY_DINH_CHI_TIET': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'QUY ĐỊNH CHI TIẾT, HƯỚNG DẪN THI HÀNH': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'QUY ĐỊNH CHI TIẾT / HƯỚNG DẪN THI HÀNH': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'VB_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'ĐƯỢC QUY ĐỊNH CHI TIẾT, HƯỚNG DẪN THI HÀNH': 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'ĐƯỢC QUY ĐỊNH CHI TIẾT / HƯỚNG DẪN THI HÀNH BỞI': 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'VB_DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN': 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN', 'HET_HIEU_LUC_MOT_PHAN': 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN', 'QUY ĐỊNH VỀ VIỆC HẾT HIỆU LỰC MỘT PHẦN': 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN', 'LÀM HẾT HIỆU LỰC MỘT PHẦN': 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN', 'VB_QUY_DINH_HET_HIEU_LUC_MOT_PHAN': 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN', 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN': 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN', 'BỊ TÁC ĐỘNG HIỆU LỰC MỘT PHẦN': 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN', 'BỊ LÀM HẾT HIỆU LỰC MỘT PHẦN BỞI': 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN', 'VB_BI_TAC_DONG_HIEU_LUC_MOT_PHAN': 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN', 'QUY_DINH_HET_HIEU_LUC_TOAN_BO': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'HET_HIEU_LUC_TOAN_BO': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'HET_HIEU_LUC': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'QUY ĐỊNH VỀ VIỆC HẾT HIỆU LỰC TOÀN BỘ': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'LÀM HẾT HIỆU LỰC TOÀN BỘ': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'VB_QUY_DINH_HET_HIEU_LUC_TOAN_BO': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'BI_TAC_DONG_HIEU_LUC_TOAN_BO': 'BI_TAC_DONG_HIEU_LUC_TOAN_BO', 'TAC_DONG_HIEU_LUC_TOAN_BO': 'BI_TAC_DONG_HIEU_LUC_TOAN_BO', 'BỊ TÁC ĐỘNG HIỆU LỰC TOÀN BỘ': 'BI_TAC_DONG_HIEU_LUC_TOAN_BO', 'BỊ LÀM HẾT HIỆU LỰC TOÀN BỘ BỞI': 'BI_TAC_DONG_HIEU_LUC_TOAN_BO', 'VB_BI_TAC_DONG_HIEU_LUC_TOAN_BO': 'BI_TAC_DONG_HIEU_LUC_TOAN_BO', 'DINH_CHINH': 'DINH_CHINH', 'ĐÍNH CHÍNH': 'DINH_CHINH', 'VB_DINH_CHINH': 'DINH_CHINH', 'DUOC_DINH_CHINH': 'DUOC_DINH_CHINH', 'ĐƯỢC ĐÍNH CHÍNH': 'DUOC_DINH_CHINH', 'ĐƯỢC ĐÍNH CHÍNH BỞI': 'DUOC_DINH_CHINH', 'VB_DUOC_DINH_CHINH': 'DUOC_DINH_CHINH', 'DAN_CHIEU': 'DAN_CHIEU', 'DẪN CHIẾU': 'DAN_CHIEU', 'DẪN CHIẾU ĐẾN': 'DAN_CHIEU', 'VB_DAN_CHIEU': 'DAN_CHIEU', 'DUOC_DAN_CHIEU': 'DUOC_DAN_CHIEU', 'ĐƯỢC DẪN CHIẾU': 'DUOC_DAN_CHIEU', 'ĐƯỢC DẪN CHIẾU BỞI': 'DUOC_DAN_CHIEU', 'VB_DUOC_DAN_CHIEU': 'DUOC_DAN_CHIEU', 'AP_DUNG': 'AP_DUNG', 'ÁP DỤNG': 'AP_DUNG', 'VB_AP_DUNG': 'AP_DUNG', 'GIAI_THICH': 'GIAI_THICH', 'GIẢI THÍCH': 'GIAI_THICH', 'GIẢI THÍCH CHO': 'GIAI_THICH', 'VB_GIAI_THICH': 'GIAI_THICH', 'DUOC_GIAI_THICH': 'DUOC_GIAI_THICH', 'ĐƯỢC GIẢI THÍCH': 'DUOC_GIAI_THICH', 'ĐƯỢC GIẢI THÍCH BỞI': 'DUOC_GIAI_THICH', 'VB_DUOC_GIAI_THICH': 'DUOC_GIAI_THICH', 'HOP_NHAT': 'HOP_NHAT', 'HỢP NHẤT': 'HOP_NHAT', 'HỢP NHẤT VÀO / HỢP NHẤT': 'HOP_NHAT', 'VB_HOP_NHAT': 'HOP_NHAT', 'DINH_CHI_THI_HANH': 'DINH_CHI_THI_HANH', 'ĐÌNH CHỈ THI HÀNH': 'DINH_CHI_THI_HANH', 'VB_DINH_CHI_THI_HANH': 'DINH_CHI_THI_HANH', 'BI_DINH_CHI_THI_HANH': 'BI_DINH_CHI_THI_HANH', 'BỊ ĐÌNH CHỈ THI HÀNH': 'BI_DINH_CHI_THI_HANH', 'BỊ ĐÌNH CHỈ THI HÀNH BỞI': 'BI_DINH_CHI_THI_HANH', 'VB_BI_DINH_CHI_THI_HANH': 'BI_DINH_CHI_THI_HANH', 'QUY_DINH_TAM_NGUNG_GIA_HAN': 'QUY_DINH_TAM_NGUNG_GIA_HAN', 'TẠM NGƯNG / GIA HẠN / CÔNG BỐ TIẾP TỤC HIỆU LỰC': 'QUY_DINH_TAM_NGUNG_GIA_HAN', 'VB_QUY_DINH_TAM_NGUNG_GIA_HAN': 'QUY_DINH_TAM_NGUNG_GIA_HAN', 'BI_TAM_NGUNG_GIA_HAN': 'BI_TAM_NGUNG_GIA_HAN', 'BỊ TẠM NGƯNG / GIA HẠN': 'BI_TAM_NGUNG_GIA_HAN', 'BỊ TẠM NGƯNG / ĐƯỢC GIA HẠN / ĐƯỢC CÔNG BỐ TIẾP TỤC HIỆU LỰC BỞI': 'BI_TAM_NGUNG_GIA_HAN', 'VB_BI_TAM_NGUNG_GIA_HAN': 'BI_TAM_NGUNG_GIA_HAN'}

def normalize_relation_label(rel: str) -> str:
    """Chuẩn hóa mọi biểu diễn nhãn quan hệ sang mã UPPERCASE chuẩn trong ALL_RELATIONS."""
    if not rel:
        return 'THUOC'
    clean = str(rel).strip().upper()
    if clean in RELATION_ALIAS_MAP:
        return RELATION_ALIAS_MAP[clean]
    clean_lower = clean.lower()
    for k, v in RELATION_ALIAS_MAP.items():
        if k.lower() == clean_lower:
            return v
    if clean in ALL_RELATIONS:
        return clean
    return 'THUOC'

def normalize_entity_label(label: str) -> str:
    """Chuẩn hóa tên thực thể sang mã UPPERCASE chuẩn trong ENTITY_TAXONOMY."""
    if not label:
        return 'DIEU'
    clean = str(label).strip().upper()
    if clean in ENTITY_TAXONOMY:
        return clean
    for code, info in ENTITY_TAXONOMY.items():
        if info[1].strip().upper() == clean:
            return code
    return clean
INVERSE_RELATION_MAP: Dict[str, str] = {'THUOC': 'THUOC', 'HUONG_DAN_AP_DUNG': 'DUOC_HUONG_DAN_AP_DUNG', 'DUOC_HUONG_DAN_AP_DUNG': 'HUONG_DAN_AP_DUNG', 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN': 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN', 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN': 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN', 'QUY_DINH_HET_HIEU_LUC_TOAN_BO': 'BI_TAC_DONG_HIEU_LUC_TOAN_BO', 'BI_TAC_DONG_HIEU_LUC_TOAN_BO': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'DINH_CHINH': 'DUOC_DINH_CHINH', 'DUOC_DINH_CHINH': 'DINH_CHINH', 'DAN_CHIEU': 'DUOC_DAN_CHIEU', 'DUOC_DAN_CHIEU': 'DAN_CHIEU', 'CAN_CU_BAN_HANH': 'AP_DUNG', 'AP_DUNG': 'CAN_CU_BAN_HANH', 'GIAI_THICH': 'DUOC_GIAI_THICH', 'DUOC_GIAI_THICH': 'GIAI_THICH', 'HOP_NHAT': 'HOP_NHAT', 'DINH_CHI_THI_HANH': 'BI_DINH_CHI_THI_HANH', 'BI_DINH_CHI_THI_HANH': 'DINH_CHI_THI_HANH', 'QUY_DINH_TAM_NGUNG_GIA_HAN': 'BI_TAM_NGUNG_GIA_HAN', 'BI_TAM_NGUNG_GIA_HAN': 'QUY_DINH_TAM_NGUNG_GIA_HAN'}
SYMMETRIC_RELATIONS: Set[str] = {'HOP_NHAT'}

def get_inverse_relation(relation: str) -> Optional[str]:
    norm = normalize_relation_label(relation)
    return INVERSE_RELATION_MAP.get(norm)

def is_symmetric_relation(relation: str) -> bool:
    norm = normalize_relation_label(relation)
    return norm in SYMMETRIC_RELATIONS

def is_strictly_directed(relation: str) -> bool:
    return not is_symmetric_relation(relation)
LABEL_XML_ENTITY_MAP = {info[1]: code for code, info in ENTITY_TAXONOMY.items()}
LABEL_XML_RELATION_MAP = {info[1]: code for code, info in RELATION_TAXONOMY.items()}
CODE_TO_LABEL_XML_ENTITY = {code: info[1] for code, info in ENTITY_TAXONOMY.items()}
CODE_TO_LABEL_XML_RELATION = {code: info[1] for code, info in RELATION_TAXONOMY.items()}

print(f'✅ Schema: {len(ALL_ENTITIES)} loại thực thể, {len(ALL_RELATIONS)} loại quan hệ.')


### 🕸️ [BƯỚC 5] Data Pipeline, Kiểm tra Dataset & PhoBERT Sliding Window
Đọc JSON trực tiếp trong cell; lỗi hiện đầy đủ tên file và nguyên nhân, không bị bọc bởi
`CalledProcessError`. CSV là bản xuất trùng, không được đếm thêm thành tài liệu.

Loại phụ lục tự sinh chứa đáp án; chuẩn hóa khoảng trắng ranh giới và kiểm tra offset.
PyVi tách từ; cửa sổ chia theo BPE token, tối đa 254 token nội dung + BOS/EOS.
Xem `data_audit.json` trước khi train. Dataset gốc không bị sửa.


In [ ]:
"""Validated raw-document inputs; labels are never graph features."""
import hashlib
import json
import random
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
FOOTER_MARKER = '--- THÔNG TIN THUỘC TÍNH & LƯỢC ĐỒ TỪ FILE JSON ---'
PIPELINE_VERSION = 'raw-doc-v2'

def normalized_body_hash(text):
    return digest(' '.join(unicodedata.normalize('NFC', body_text(text)).split()))

def body_text(text):
    return text.split(FOOTER_MARKER, 1)[0]

def digest(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

def load_documents(root):
    """Require JSON full_text; CSV duplicates are not separate documents.

    Keep exact original offsets. Invalid/ambiguous spans are reported, not guessed.
    """
    root = Path(root)
    paths = sorted(root.rglob('*_ket_qua.json'))
    if not paths:
        raise ValueError(f'No *_ket_qua.json with full_text in {root}')
    documents, issues, totals = ([], [], Counter())
    entity_counts, relation_counts, relation_docs = (Counter(), Counter(), Counter())
    for path in paths:
        try:
            raw = json.loads(path.read_text(encoding='utf-8-sig'))
        except (OSError, ValueError) as exc:
            raise ValueError(f'Cannot read dataset JSON {path}: {exc}') from exc
        if isinstance(raw, list):
            if len(raw) != 1:
                raise ValueError(f'Expected one document: {path}')
            raw = raw[0]
        if not isinstance(raw, dict):
            raise ValueError(f'Expected a document object: {path}')
        if not isinstance(raw.get('entities', []), list) or not isinstance(raw.get('relations', []), list):
            raise ValueError(f'entities and relations must be lists: {path}')
        if not isinstance(raw.get('full_text'), str) or not raw['full_text'].strip():
            raise ValueError(f'Missing full_text: {path}')
        original = raw['full_text']
        text = body_text(original)
        if not text.strip():
            raise ValueError(f'Missing original document body: {path}')
        totals['documents'] += 1
        totals['documents_with_synthetic_footer'] += FOOTER_MARKER in original
        entities, ids = ([], set())
        for entity in raw.get('entities', []):
            totals['input_entities'] += 1
            if not isinstance(entity, dict) or not entity.get('id'):
                raise ValueError(f'Entity missing ID or invalid entity object: {path}')
            eid = entity['id']
            if eid in ids:
                raise ValueError(f'Duplicate entity ID {eid}: {path}')
            ids.add(eid)
            code = entity.get('code')
            if code not in ALL_ENTITIES:
                raise ValueError(f'Unknown entity code {code}: {path}')
            start, end = (entity.get('start'), entity.get('end'))
            if not isinstance(start, int) or not isinstance(end, int):
                reason = 'invalid_offsets'
            elif start >= len(text) and start < len(original):
                reason = 'synthetic_footer'
            elif not 0 <= start < end <= len(text):
                reason = 'invalid_or_cross_footer_offsets'
            elif text[start:end] != entity.get('text'):
                reason = 'text_offset_mismatch'
            elif not entity['text'].strip():
                reason = 'empty_entity_text'
            else:
                literal = entity['text']
                clean_start = start + len(literal) - len(literal.lstrip())
                clean_end = end - (len(literal) - len(literal.rstrip()))
                if (clean_start, clean_end) != (start, end):
                    totals['entities_whitespace_boundary_trimmed'] += 1
                    issues.append({'file': str(path.relative_to(root)), 'entity_id': eid, 'reason': 'normalized_surrounding_whitespace', 'original_span': [start, end], 'normalized_span': [clean_start, clean_end]})
                entities.append(dict(entity, start=clean_start, end=clean_end, text=text[clean_start:clean_end]))
                entity_counts[code] += 1
                continue
            totals[f'excluded_entities_{reason}'] += 1
            issues.append({'file': str(path.relative_to(root)), 'entity_id': eid, 'reason': reason})
        kept_ids = {e['id'] for e in entities}
        relations, seen = ([], set())
        for relation in raw.get('relations', []):
            totals['input_relations'] += 1
            if not isinstance(relation, dict):
                raise ValueError(f'Invalid relation object: {path}')
            code = str(relation.get('code') or relation.get('label') or '').strip().upper()
            if code not in RELATION_ALIAS_MAP:
                raise ValueError(f'Unknown relation code {code}: {path}')
            canonical = RELATION_ALIAS_MAP[code]
            key = (relation.get('from_id'), relation.get('to_id'), canonical)
            if key[0] not in ids or key[1] not in ids:
                raise ValueError(f'Dangling relation {key}: {path}')
            if key[0] not in kept_ids or key[1] not in kept_ids:
                totals['excluded_relations_missing_body_entity'] += 1
                continue
            if key in seen:
                totals['duplicate_relations_removed'] += 1
                continue
            seen.add(key)
            relations.append({'from_id': key[0], 'to_id': key[1], 'code': canonical})
            relation_counts[canonical] += 1
        relation_docs.update({r['code'] for r in relations})
        documents.append({'id': path.relative_to(root).as_posix(), 'text': text, 'entities': entities, 'relations': relations})
    totals['retained_entities'] = sum(entity_counts.values())
    totals['retained_relations'] = sum(relation_counts.values())
    report = {'version': PIPELINE_VERSION, 'totals': dict(totals), 'entity_counts': {k: entity_counts[k] for k in ALL_ENTITIES}, 'relation_counts': {k: relation_counts[k] for k in ALL_RELATIONS}, 'relation_document_support': {k: relation_docs[k] for k in ALL_RELATIONS}, 'issues': issues, 'note': 'Supplied annotations are not certified as human-reviewed gold.'}
    return (documents, report)

def split_documents(documents, output, seed=42):
    """Persist splits; identical normalized bodies stay together; reject stale manifests."""
    output = Path(output)
    ids = [d['id'] for d in documents]
    if len(ids) != len(set(ids)):
        raise ValueError('Duplicate document IDs: use relative paths, not basenames.')
    fingerprints = {d['id']: digest(json.dumps(d, sort_keys=True, ensure_ascii=False)) for d in documents}
    if output.exists():
        manifest = json.loads(output.read_text(encoding='utf-8'))
        if manifest.get('version') != PIPELINE_VERSION:
            raise ValueError('Old pipeline manifest: choose a new output directory.')
        if manifest['fingerprints'] != fingerprints or manifest['seed'] != seed:
            raise ValueError('Dataset/seed changed: choose a new output directory.')
    else:
        groups = defaultdict(list)
        for doc in documents:
            groups[digest(' '.join(unicodedata.normalize('NFC', doc['text']).split()))].append(doc['id'])
        keys = sorted(groups)
        if len(keys) < 3:
            raise ValueError('Need >=3 distinct bodies for train/val/test; sample_data is smoke-test only.')
        random.Random(seed).shuffle(keys)
        nv = max(1, int(len(keys) * 0.1))
        nt = max(1, int(len(keys) * 0.1))
        partitions = {'val': keys[:nv], 'test': keys[nv:nv + nt], 'train': keys[nv + nt:]}
        manifest = {'version': PIPELINE_VERSION, 'seed': seed, 'ratios': {'train': 0.8, 'val': 0.1, 'test': 0.1}, 'body_hashes': {d['id']: normalized_body_hash(d['text']) for d in documents}, 'fingerprints': fingerprints, 'splits': {k: [i for group in v for i in groups[group]] for k, v in partitions.items()}}
        save_json(output, manifest)
    by_id = {d['id']: d for d in documents}
    partitions = manifest['splits']
    if set(partitions) != {'train', 'val', 'test'}:
        raise ValueError('Invalid split manifest: require train/val/test')
    assigned = [i for part in partitions.values() for i in part]
    if len(assigned) != len(set(assigned)) or set(assigned) != set(by_id):
        raise ValueError('Split overlap/missing documents: choose a new output directory.')
    hashes = []
    for name in ('train', 'val', 'test'):
        if not partitions[name]:
            raise ValueError('Each split must contain at least one document.')
        hashes.append({normalized_body_hash(by_id[i]['text']) for i in partitions[name]})
    if any((a & b for n, a in enumerate(hashes) for b in hashes[n + 1:])):
        raise ValueError('Identical document bodies occur across splits.')
    return {k: [by_id[i] for i in ids] for k, ids in manifest['splits'].items()}

def atoms(text):
    return [(m.start(), m.end()) for m in re.finditer('\\w+|[^\\w\\s]', text, re.UNICODE)]

def span_key(entity):
    return (entity['start'], entity['end'], entity['code'])

def relation_gold(doc):
    by_id = {e['id']: e for e in doc['entities']}
    return {(span_key(by_id[r['from_id']]), span_key(by_id[r['to_id']]), ALL_RELATIONS.index(r['code'])) for r in doc['relations']}

def candidate_pairs(entities, neighbors=8, anchors=12, enclosing=0):
    """Offset-only directed neighbors and earliest anchors; O(N*(neighbors+anchors)).

    No target labels, IDs, or relation-encoded entity types are consulted.
    Gold relations outside this candidate set must still count as false negatives.
    """
    order = sorted(range(len(entities)), key=lambda i: (entities[i]['start'], entities[i]['end'], i))
    pairs = set()
    for rank, i in enumerate(order):
        for j in order[max(0, rank - neighbors):rank + neighbors + 1]:
            if i != j:
                pairs.add((i, j))
    for i in order[:anchors]:
        for j in order:
            if i != j:
                pairs.update(((i, j), (j, i)))
    if enclosing:
        active = []
        containment_order = sorted(order, key=lambda i: (entities[i]['start'], -entities[i]['end']))
        for i in containment_order:
            e = entities[i]
            active = [j for j in active if entities[j]['end'] >= e['end'] or entities[j]['end'] > e['start']]
            parents = [j for j in active if entities[j]['end'] >= e['end'] and (entities[j]['start'], entities[j]['end']) != (e['start'], e['end'])]
            parents.sort(key=lambda j: (entities[j]['end'] - entities[j]['start'], j))
            for j in parents[:enclosing]:
                pairs.update(((i, j), (j, i)))
            active.append(i)
    return sorted(pairs)

def topology_edges(entities):
    """Observable types: adjacency forward/backward and overlap forward/backward."""
    order = sorted(range(len(entities)), key=lambda i: (entities[i]['start'], entities[i]['end'], i))
    edges, active = (set(), [])
    for rank, i in enumerate(order):
        if rank:
            j = order[rank - 1]
            edges.update(((j, i, 0), (i, j, 1)))
        active = [j for j in active if entities[j]['end'] > entities[i]['start']]
        for j in active:
            edges.update(((j, i, 2), (i, j, 3)))
            if len(edges) > 500000:
                raise ValueError('Graph exceeds 500,000 positional edges. Inspect excessive/overlapping NER spans or use shorter documents; edges are not silently dropped.')
        active.append(i)
    return sorted(edges)

def set_scores(gold, predicted):
    tp = len(gold & predicted)
    p = tp / len(predicted) if predicted else 0.0
    r = tp / len(gold) if gold else 0.0
    return {'precision': p, 'recall': r, 'f1': 2 * p * r / (p + r) if p + r else 0.0, 'tp': tp, 'fp': len(predicted) - tp, 'fn': len(gold) - tp}

"""Frozen PhoBERT, Vietnamese segmentation, token windows and original offsets."""
import importlib.metadata
import re
from pathlib import Path
import torch
from pyvi import ViTokenizer
from transformers import AutoModel, AutoTokenizer

def segmented_words(text):
    """Align PyVi words back to the unchanged input; never infer annotation offsets."""
    result = []
    for line in re.finditer('[^\\r\\n]+', text):
        cursor = line.start()
        for word in ViTokenizer.tokenize(line.group()).split():
            literal_start = cursor
            while literal_start < line.end() and text[literal_start].isspace():
                literal_start += 1
            if text.startswith(word, literal_start):
                cursor = literal_start + len(word)
                result.append((word, literal_start, cursor))
                continue
            start = None
            for part in word.split('_'):
                if not part:
                    continue
                match = text.find(part, cursor, line.end())
                if match < 0 or any((not c.isspace() and c != '_' for c in text[cursor:match])):
                    raise ValueError(f'Cannot align segmented word {word!r} at character {cursor}; inspect raw text.')
                start = match if start is None else start
                cursor = match + len(part)
            if start is not None:
                result.append((word, start, cursor))
        if text[cursor:line.end()].strip():
            raise ValueError(f'Segmentation dropped input near character {cursor}')
    return result

def aligned_units(text, words):
    """Split raw lexical units when PyVi splits inside them (e.g. '2,5a')."""
    units = []
    index = 0
    for start, end in atoms(text):
        cursor = start
        while cursor < end:
            while index + 1 < len(words) and words[index][2] <= cursor:
                index += 1
            if not words or not words[index][1] <= cursor < words[index][2]:
                raise ValueError(f'No segmented word covers character {cursor}')
            stop = min(end, words[index][2])
            units.append((cursor, stop))
            cursor = stop
    return units

class ContextEncoder:

    def __init__(self, model_name='vinai/phobert-base-v2', window=256, stride=128, batch_size=8, device=None, revision='main', local_files_only=False):
        if not 4 <= window <= 256 or not 1 <= stride <= window - 2:
            raise ValueError('Require window<=256 and 1<=stride<=window-2')
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, revision=revision, local_files_only=local_files_only)
        self.model = AutoModel.from_pretrained(model_name, revision=revision, local_files_only=local_files_only).to(self.device).eval()
        self.model.requires_grad_(False)
        if self.model.config.hidden_size != 768:
            raise ValueError('This pipeline expects a 768-dimensional PhoBERT encoder')
        self.window, self.stride, self.batch_size = (window, stride, batch_size)
        self.config = {'version': PIPELINE_VERSION, 'model_name': model_name, 'revision': getattr(self.model.config, '_commit_hash', None) or revision, 'window': window, 'stride': stride, 'pyvi_version': importlib.metadata.version('pyvi'), 'transformers_version': importlib.metadata.version('transformers')}

    @torch.no_grad()
    def encode(self, text, cache_dir=None):
        key = digest(str(sorted(self.config.items())) + text)
        path = Path(cache_dir) / f'{key}.pt' if cache_dir else None
        if path and path.exists():
            return torch.load(path, map_location='cpu', weights_only=True)
        words = segmented_words(text)
        units = aligned_units(text, words)
        if not words or not units:
            return {'x': torch.empty(0, 772), 'offsets': units, 'cache_key': key}
        token_ids, owners = ([], [])
        for i, (word, _, _) in enumerate(words):
            ids = self.tokenizer.encode(word, add_special_tokens=False)
            if not ids:
                ids = [self.tokenizer.unk_token_id]
            token_ids.extend(ids)
            owners.extend([i] * len(ids))
        capacity = self.window - 2
        starts = []
        for start in range(0, len(token_ids), self.stride):
            starts.append(start)
            if start + capacity >= len(token_ids):
                break
        sums = torch.zeros(len(words), 768)
        counts = torch.zeros(len(words), 1)
        for first in range(0, len(starts), self.batch_size):
            positions = starts[first:first + self.batch_size]
            chunks = [token_ids[s:s + capacity] for s in positions]
            width = max(map(len, chunks)) + 2
            ids = torch.full((len(chunks), width), self.tokenizer.pad_token_id, dtype=torch.long)
            mask = torch.zeros_like(ids)
            for i, chunk in enumerate(chunks):
                seq = [self.tokenizer.bos_token_id] + chunk + [self.tokenizer.eos_token_id]
                ids[i, :len(seq)] = torch.tensor(seq)
                mask[i, :len(seq)] = 1
            out = self.model(input_ids=ids.to(self.device), attention_mask=mask.to(self.device)).last_hidden_state.cpu()
            for i, (start, chunk) in enumerate(zip(positions, chunks)):
                idx = torch.tensor(owners[start:start + len(chunk)])
                sums.index_add_(0, idx, out[i, 1:1 + len(chunk)])
                counts.index_add_(0, idx, torch.ones(len(chunk), 1))
        means = sums / counts.clamp_min(1)
        features = torch.empty(len(units), 772)
        word_idx = 0
        for i, (a, b) in enumerate(units):
            while word_idx + 1 < len(words) and words[word_idx][2] <= a:
                word_idx += 1
            _, wa, wb = words[word_idx]
            if not wa <= a < b <= wb:
                raise ValueError(f'Lexical unit {(a, b)} not covered by segmented words')
            features[i, :768] = means[word_idx]
            features[i, 768:] = torch.tensor([(a - wa) / max(1, wb - wa), (b - wa) / max(1, wb - wa), a / max(1, len(text)), min(b - a, 100) / 100])
        result = {'x': features.half(), 'offsets': units, 'cache_key': key}
        if path:
            path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(result, path)
        return result

# AUDIT DATASET
import pandas as pd
from IPython.display import display

if globals().get('TRAIN_NEW_MODEL', True):
    if (OUTPUT_DIR/'run_config.json').exists():
        raise ValueError('OUTPUT_DIR already contains a run; choose a new directory before auditing/training.')
    documents, audit = load_documents(DATASET_DIR)
    save_json(OUTPUT_DIR/'data_audit.json', audit)
    display(pd.DataFrame.from_dict(audit['totals'], orient='index', columns=['Số lượng']))
    display(pd.DataFrame({'Số quan hệ': audit['relation_counts'], 'Số văn bản': audit['relation_document_support']}))
    supported = sum(n > 0 for n in audit['relation_counts'].values())
    print(f'✅ Đã đọc {len(documents)} văn bản. Trần Macro-F1 cố định 22 lớp theo hỗ trợ nhãn: {supported/22:.2%}.')
    preview_splits = split_documents(documents, OUTPUT_DIR/'split_manifest.json', seed=globals().get("TRAIN_SEED",42))
    display(pd.DataFrame([{'Tập':k,'Số văn bản':len(v),'Số loại quan hệ có mẫu':len({r['code'] for d in v for r in d['relations']})} for k,v in preview_splits.items()]))
    print('Kiểm tra bảng nhãn hiếm trước khi train. Split nhóm bản trùng chính xác; cần rà soát thêm bản gần trùng và chất lượng nhãn.')
else:
    print('Đã nạp định nghĩa. Chế độ kiểm tra: giữ nguyên dữ liệu và checkpoint đã lưu.')


### 🧠 [BƯỚC 6] Khởi tạo Kiến trúc Mô hình Lai
PhoBERT frozen → NER start/end → nút thực thể có ngữ cảnh → RGCN 2 tầng → Gated Fusion →
phân loại quan hệ đa nhãn. Cạnh GNN chỉ dùng kề nhau/chồng lấn theo vị trí, không dùng đáp án.
Baseline `semantic` dùng cùng đặc trưng nhưng không có GNN.


In [ ]:
"""Shared training/inference models. GNN edges are observable, never gold labels."""
import torch
from torch import nn
from torch_geometric.nn import RGCNConv
FEATURE_DIM = 772
NUM_TOPOLOGY_RELATIONS = 4

class BoundaryNER(nn.Module):
    """Independent boundaries permit overlapping entity types, unlike flat BIO."""

    def __init__(self, hidden=128, dropout=0.2):
        super().__init__()
        self.head = nn.Sequential(nn.Linear(FEATURE_DIM, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 2 * len(ALL_ENTITIES)))

    def forward(self, x):
        return self.head(x)

class LegalRGCN(nn.Module):

    def __init__(self, hidden_channels=128, dropout=0.2, use_gnn=True):
        super().__init__()
        self.use_gnn = use_gnn
        self.node_proj = nn.Sequential(nn.Linear(FEATURE_DIM, hidden_channels), nn.LayerNorm(hidden_channels), nn.GELU())
        self.convs = nn.ModuleList([RGCNConv(hidden_channels, hidden_channels, 4, num_bases=4) for _ in range(2)]) if use_gnn else nn.ModuleList()
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_channels) for _ in self.convs])
        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Linear(hidden_channels * 2, hidden_channels)
        self.classifier = nn.Sequential(nn.Linear(hidden_channels * 4, hidden_channels), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_channels, len(ALL_RELATIONS)))

    def encode_nodes(self, x, edge_index, edge_type):
        semantic = self.node_proj(x)
        h = semantic
        for conv, norm in zip(self.convs, self.norms):
            h = self.dropout(torch.nn.functional.gelu(norm(conv(h, edge_index, edge_type))))
        if self.use_gnn:
            gate = torch.sigmoid(self.gate(torch.cat([semantic, h], dim=-1)))
            h = gate * semantic + (1 - gate) * h
        return h

    def classify(self, h, pairs):
        u, v = (h[pairs[0]], h[pairs[1]])
        return self.classifier(torch.cat([u, v, abs(u - v), u * v], dim=-1))

    def forward(self, x, edge_index, edge_type, pairs):
        return self.classify(self.encode_nodes(x, edge_index, edge_type), pairs)

preview_model = LegalRGCN(hidden_channels=128)
print(f'✅ RGCN: {sum(p.numel() for p in preview_model.parameters()):,} tham số.')
del preview_model


### 🏋️ [BƯỚC 7] Chia Train/Val/Test & Vòng lặp Huấn luyện
Sửa cấu hình ở đầu cell. Cần ít nhất 3 thân văn bản khác nhau; không nhân bản dữ liệu mẫu.
Split cố định theo tài liệu; validation/test dùng toàn bộ cặp ứng viên cố định. NER chọn
checkpoint theo exact-span F1. RE chọn theo micro-F1 validation dùng thực thể NER dự đoán.
Cặp gold ngoài ứng viên vẫn tính FN. Cặp bao chứa được bổ sung từ offset, không đọc nhãn.

Mỗi lần huấn luyện dùng `OUTPUT_DIR` mới để không ghi đè kết quả. Để kiểm tra DOCX với mô hình
đã lưu: chạy các bước định nghĩa như hướng dẫn README, không chạy lại Bước 7.
Nếu hết GPU RAM, giảm `ENCODER_BATCH` xuống 2.


In [ ]:
# CẤU HÌNH HUẤN LUYỆN — chỉ cần sửa ở đây
SEED = globals().get('TRAIN_SEED',42)
NER_EPOCHS = 20
RELATION_EPOCHS = 30
ENCODER = 'vinai/phobert-base-v2'
ENCODER_BATCH = 8
NEGATIVE_RATIO = 5.0
NEIGHBORS = 8
ANCHORS = 12
ENCLOSING_CANDIDATES = 2
LEARNING_RATE = 1e-3
DROPOUT = 0.2
VARIANTS = 'semantic,rgcn'
LOCAL_FILES_ONLY = False  # Chỉ bật nếu đã có PhoBERT trong cache local.
globals().pop('extractor', None)  # Nạp lại checkpoint nếu train lại trong cùng phiên.

"""Reproducible frozen-PhoBERT NER + relation training on raw legal documents.

Run --audit-only before training. Test labels are used only for scoring.
"""
import argparse
import bisect
import csv
import json
import random
import importlib.metadata
import time
from collections import Counter
from pathlib import Path
import numpy as np
import torch
from torch import nn

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def read_features(path):
    return torch.load(path, map_location='cpu', weights_only=True)

def boundary_targets(doc, offsets):
    y = torch.zeros(len(offsets), 2 * len(ALL_ENTITIES))
    starts = {a: i for i, (a, b) in enumerate(offsets)}
    ends = {b: i for i, (a, b) in enumerate(offsets)}
    unsupported = []
    for e in doc['entities']:
        if e['start'] not in starts or e['end'] not in ends:
            unsupported.append(e['id'])
            continue
        code = ALL_ENTITIES.index(e['code'])
        y[starts[e['start']], code] = 1
        y[ends[e['end']], code + len(ALL_ENTITIES)] = 1
    return (y, unsupported)

def decode_entities(probabilities, offsets, text, threshold=0.5):
    """Nearest following end for each start/type; can cross encoder windows.

    Cross-type overlap is supported. Same-type nested spans sharing boundaries
    cannot be represented unambiguously; exact-span evaluation exposes this limit.
    """
    entities, seen = ([], set())
    ntypes = len(ALL_ENTITIES)
    for code_idx, code in enumerate(ALL_ENTITIES):
        starts = torch.where(probabilities[:, code_idx] >= threshold)[0].tolist()
        ends = torch.where(probabilities[:, code_idx + ntypes] >= threshold)[0].tolist()
        for start in starts:
            loc = bisect.bisect_left(ends, start)
            if loc == len(ends):
                continue
            end = ends[loc]
            a, b = (offsets[start][0], offsets[end][1])
            if (a, b, code) not in seen:
                seen.add((a, b, code))
                entities.append({'start': a, 'end': b, 'code': code, 'text': text[a:b], 'score': float(min(probabilities[start, code_idx], probabilities[end, code_idx + ntypes]))})
    entities.sort(key=lambda e: (e['start'], e['end'], e['code']))
    for i, e in enumerate(entities):
        e['id'] = f'P{i}'
    return entities

@torch.no_grad()
def predict_entities(model, feature, text, device, threshold, batch_size=4096):
    model.eval()
    x = feature['x']
    probs = [model(x[i:i + batch_size].float().to(device)).sigmoid().cpu() for i in range(0, len(x), batch_size)]
    return decode_entities(torch.cat(probs) if probs else torch.empty(0, 2 * len(ALL_ENTITIES)), feature['offsets'], text, threshold)

@torch.no_grad()
def evaluate_ner(model, docs, features, device, threshold):
    gold, pred = (set(), set())
    for doc in docs:
        feature = read_features(features[doc['id']])
        inferred = predict_entities(model, feature, doc['text'], device, threshold)
        gold.update(((doc['id'], *span_key(e)) for e in doc['entities']))
        pred.update(((doc['id'], *span_key(e)) for e in inferred))
    scores = set_scores(gold, pred)
    scores['per_type'] = {code: set_scores({e for e in gold if e[-1] == code}, {e for e in pred if e[-1] == code}) for code in ALL_ENTITIES}
    return scores

def checkpoint_payload(model, config, **extra):
    return {'version': PIPELINE_VERSION, 'model_state_dict': model.state_dict(), 'config': config, 'entity_labels': ALL_ENTITIES, 'relation_labels': ALL_RELATIONS, **extra}

def train_ner(splits, features, config, output, device):
    set_seed(config['seed'])
    model = BoundaryNER(config['hidden'], config['dropout']).to(device)
    counts = torch.zeros(2 * len(ALL_ENTITIES))
    total = 0
    alignment = {}
    train_ids = {doc['id'] for doc in splits['train']}
    for doc in sum(splits.values(), []):
        feature = read_features(features[doc['id']])
        y, unsupported = boundary_targets(doc, feature['offsets'])
        if unsupported:
            alignment[doc['id']] = unsupported
        if doc['id'] in train_ids:
            counts += y.sum(0)
            total += len(y)
    save_json(output / 'boundary_alignment_issues.json', alignment)
    pos_weight = ((total - counts) / counts.clamp_min(1)).clamp(1, 50).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=0.01)
    best, stale, history = (-1.0, 0, [])
    best_path = output / 'checkpoints' / 'ner.pt'
    best_path.parent.mkdir(parents=True, exist_ok=True)
    for epoch in range(config['ner_epochs']):
        model.train()
        order = list(splits['train'])
        random.Random(config['seed'] + epoch).shuffle(order)
        total_loss, n = (0.0, 0)
        for doc in order:
            feature = read_features(features[doc['id']])
            y, _ = boundary_targets(doc, feature['offsets'])
            indices = torch.randperm(len(y))
            for start in range(0, len(indices), config['token_batch']):
                idx = indices[start:start + config['token_batch']]
                x = feature['x'][idx].float().to(device)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(x), y[idx].to(device))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item() * len(idx)
                n += len(idx)
        validation = [(threshold, evaluate_ner(model, splits['val'], features, device, threshold)) for threshold in config['thresholds']]
        threshold, scores = max(validation, key=lambda item: item[1]['f1'])
        history.append({'epoch': epoch + 1, 'loss': total_loss / max(1, n), 'val_f1': scores['f1'], 'threshold': threshold})
        print(f"NER {epoch + 1}: loss={history[-1]['loss']:.4f}, val exact-span F1={scores['f1']:.4f}", flush=True)
        if scores['f1'] > best:
            best, stale = (scores['f1'], 0)
            torch.save(checkpoint_payload(model, config, threshold=threshold, epoch=epoch + 1, validation=scores, kind='ner'), best_path)
        else:
            stale += 1
        save_json(output / 'ner_history.json', history)
        if stale >= config['patience']:
            break
    ckpt = load_checkpoint(best_path, device)
    model.load_state_dict(ckpt['model_state_dict'], strict=True)
    return (model.eval(), ckpt['threshold'])

def make_graph(text, entities, feature, neighbors=8, anchors=12, enclosing=0):
    """Pool contextual span+surrounding text; graph only sees observable geometry."""
    offsets = feature['offsets']
    starts, ends = ([a for a, b in offsets], [b for a, b in offsets])
    x = torch.zeros(len(entities), 772)
    for i, entity in enumerate(entities):
        a = bisect.bisect_right(ends, entity['start'])
        b = bisect.bisect_left(starts, entity['end'])
        if a >= b:
            raise ValueError(f'Entity has no lexical units: {entity}')
        span = feature['x'][a:b, :768].float().mean(0)
        context = feature['x'][max(0, a - 16):min(len(offsets), b + 16), :768].float().mean(0)
        x[i, :768] = (span + context) / 2
        x[i, 768:] = torch.tensor([entity['start'] / max(1, len(text)), entity['end'] / max(1, len(text)), min(b - a, 1000) / 1000, min(a, 1000) / 1000])
    edges = topology_edges(entities)
    ei = torch.tensor([[u, v] for u, v, r in edges], dtype=torch.long).t().contiguous() if edges else torch.empty(2, 0, dtype=torch.long)
    et = torch.tensor([r for u, v, r in edges], dtype=torch.long)
    return {'x': x, 'edge_index': ei, 'edge_type': et, 'entities': entities, 'pairs': candidate_pairs(entities, neighbors, anchors, enclosing)}

def gold_pair_labels(doc, entities):
    indices = {span_key(e): i for i, e in enumerate(entities)}
    labels = {}
    for u, v, r in relation_gold(doc):
        if u in indices and v in indices:
            labels.setdefault((indices[u], indices[v]), set()).add(r)
    return labels

def sample_training_pairs(graph, doc, seed, neg_ratio):
    positive = gold_pair_labels(doc, graph['entities'])
    negative = [pair for pair in graph['pairs'] if pair not in positive]
    rng = random.Random(seed)
    rng.shuffle(negative)
    pairs = sorted(positive) + negative[:max(1, int(max(1, len(positive)) * neg_ratio))]
    rng.shuffle(pairs)
    y = torch.zeros(len(pairs), len(ALL_RELATIONS))
    for i, pair in enumerate(pairs):
        for label in positive.get(pair, set()):
            y[i, label] = 1
    return (torch.tensor(pairs, dtype=torch.long).t().reshape(2, -1), y)

def relation_metrics(tp, fp, support, correct, candidates, covered, gold_pairs_covered):
    p = tp / (tp + fp).clip(min=1)
    r = tp / support.clip(min=1)
    f1 = 2 * p * r / (p + r).clip(min=1e-12)
    active = support > 0
    micro_p = float(tp.sum() / max(1, (tp + fp).sum()))
    micro_r = float(tp.sum() / max(1, support.sum()))
    return {'accuracy_candidates_exact_set': correct / max(1, candidates), 'precision_macro_22': float(p.mean()), 'recall_macro_22': float(r.mean()), 'f1_macro_22': float(f1.mean()), 'f1_macro_supported': float(f1[active].mean()) if active.any() else 0.0, 'precision_micro_positive': micro_p, 'recall_micro_positive': micro_r, 'f1_micro_positive': 2 * micro_p * micro_r / (micro_p + micro_r) if micro_p + micro_r else 0.0, 'candidate_recall': covered / max(1, int(support.sum())), 'candidate_count': candidates, 'no_relation_baseline_accuracy': (candidates - gold_pairs_covered) / max(1, candidates), 'supported_classes': int(active.sum()), 'per_class': {code: {'precision': float(p[i]), 'recall': float(r[i]), 'f1': float(f1[i]), 'support': int(support[i]), 'tp': int(tp[i]), 'fp': int(fp[i])} for i, code in enumerate(ALL_RELATIONS)}}

@torch.no_grad()
def evaluate_relations(model, docs, graphs, device, threshold, batch_size=2048):
    """All deterministic candidates, no sampled test negatives; missed gold counts as FN."""
    model.eval()
    tp, fp, support = (np.zeros(len(ALL_RELATIONS), dtype=np.int64) for _ in range(3))
    correct = candidates = covered = gold_pairs_covered = 0
    for doc in docs:
        graph = graphs[doc['id']]
        labels = gold_pair_labels(doc, graph['entities'])
        for u, v, r in relation_gold(doc):
            support[r] += 1
        pair_set = set(graph['pairs'])
        covered += sum((len(rs) for pair, rs in labels.items() if pair in pair_set))
        gold_pairs_covered += sum((pair in pair_set for pair in labels))
        if not graph['pairs']:
            continue
        h = model.encode_nodes(graph['x'].to(device), graph['edge_index'].to(device), graph['edge_type'].to(device))
        for first in range(0, len(graph['pairs']), batch_size):
            pairs = graph['pairs'][first:first + batch_size]
            pair_tensor = torch.tensor(pairs, dtype=torch.long, device=device).t()
            pred = (model.classify(h, pair_tensor).sigmoid() >= threshold).cpu().numpy()
            target = np.zeros_like(pred)
            for i, pair in enumerate(pairs):
                for r in labels.get(pair, set()):
                    target[i, r] = True
            tp += (pred & target).sum(0)
            fp += (pred & ~target).sum(0)
            correct += int((pred == target).all(1).sum())
            candidates += len(pairs)
    return relation_metrics(tp, fp, support, correct, candidates, covered, gold_pairs_covered)

def train_relations(splits, graphs, config, output, device, variant, validation_graphs=None):
    set_seed(config['seed'])
    model = LegalRGCN(config['hidden'], config['dropout'], variant == 'rgcn').to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=0.01)
    counts = Counter((r['code'] for d in splits['train'] for r in d['relations']))
    total = sum(counts.values())
    weights = torch.tensor([min(20.0, max(1.0, total / max(1, counts[c]) / len(ALL_RELATIONS))) for c in ALL_RELATIONS], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=weights)
    best, stale, history = (-1.0, 0, [])
    best_path = output / 'checkpoints' / f'relation_{variant}.pt'
    for epoch in range(config['relation_epochs']):
        model.train()
        order = list(splits['train'])
        random.Random(config['seed'] + epoch).shuffle(order)
        losses = []
        for i, doc in enumerate(order):
            graph = graphs[doc['id']]
            pairs, y = sample_training_pairs(graph, doc, config['seed'] + epoch * len(order) + i, config['neg_ratio'])
            if not len(y):
                continue
            optimizer.zero_grad(set_to_none=True)
            logits = model(graph['x'].to(device), graph['edge_index'].to(device), graph['edge_type'].to(device), pairs.to(device))
            loss = criterion(logits, y.to(device))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())
        validation = [(t, evaluate_relations(model, splits['val'], validation_graphs if validation_graphs is not None else graphs, device, t)) for t in config['thresholds']]
        selection_metric = config.get('selection_metric', 'f1_macro_22')
        threshold, scores = max(validation, key=lambda x: x[1][selection_metric])
        history.append({'epoch': epoch + 1, 'loss': sum(losses) / max(1, len(losses)), 'val_f1_macro_22': scores['f1_macro_22'], 'val_f1_micro_positive': scores['f1_micro_positive'], 'val_precision_micro': scores['precision_micro_positive'], 'val_recall_micro': scores['recall_micro_positive'], 'selection_metric': selection_metric, 'threshold': threshold})
        print(f"RE {variant} {epoch + 1}: loss={history[-1]['loss']:.4f}, val {selection_metric}={scores[selection_metric]:.4f}", flush=True)
        if scores[selection_metric] > best:
            best, stale = (scores[selection_metric], 0)
            torch.save(checkpoint_payload(model, config, threshold=threshold, epoch=epoch + 1, validation=scores, kind='relation', variant=variant), best_path)
        else:
            stale += 1
        save_json(output / f'relation_{variant}_history.json', history)
        if stale >= config['patience']:
            break
    ckpt = load_checkpoint(best_path, device)
    model.load_state_dict(ckpt['model_state_dict'], strict=True)
    return (model.eval(), ckpt['threshold'])

def load_checkpoint(path, device='cpu'):
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    if checkpoint.get('version') != PIPELINE_VERSION or checkpoint.get('entity_labels') != ALL_ENTITIES or checkpoint.get('relation_labels') != ALL_RELATIONS:
        raise ValueError('Incompatible checkpoint/schema; retrain using this pipeline.')
    return checkpoint

def graph_for(doc, feature, config, entities=None):
    return make_graph(doc['text'], doc['entities'] if entities is None else entities, feature, config['neighbors'], config['anchors'], config.get('enclosing', 0))

def run(args):
    output = Path(args.output).resolve()
    output.mkdir(parents=True, exist_ok=True)
    if (output / 'run_config.json').exists():
        raise ValueError('Output already contains a training run. Choose a new OUTPUT_DIR; old results stay unchanged.')
    started = time.perf_counter()
    documents, audit = load_documents(args.dataset)
    save_json(output / 'data_audit.json', audit)
    print(json.dumps(audit['totals'], ensure_ascii=False, indent=2), flush=True)
    if args.audit_only:
        return
    save_json(output / 'run_status.json', {'status': 'training', 'version': PIPELINE_VERSION})
    if min(args.ner_epochs, args.relation_epochs, args.hidden, args.token_batch, args.encoder_batch, args.patience) < 1:
        raise ValueError('Epochs, dimensions, batches and patience must be positive')
    if args.neg_ratio <= 0 or min(args.neighbors, args.anchors, args.enclosing) < 0 or (not 0 <= args.dropout < 1) or (args.lr <= 0):
        raise ValueError('Invalid sampling/model hyperparameters')
    if not args.variants or any((v not in {'semantic', 'rgcn'} for v in args.variants.split(','))):
        raise ValueError('Variants must be semantic,rgcn (or either one)')
    splits = split_documents(documents, output / 'split_manifest.json', args.seed)
    support = {name: dict(Counter((r['code'] for d in docs for r in d['relations']))) for name, docs in splits.items()}
    save_json(output / 'split_relation_support.json', support)
    for name in ('val', 'test'):
        active = len(support[name])
        print(f'{name}: {len(splits[name])} documents; {active}/22 supported relations; macro-F1 ceiling <= {active / 22:.2%}', flush=True)
        if not active:
            print(f'WARNING: {name} has no positive relations. This run can check execution but cannot establish RE quality.', flush=True)
    save_json(output / 'split_summary.json', {'counts': {name: len(docs) for name, docs in splits.items()}, 'ratios_requested': {'train': 0.8, 'val': 0.1, 'test': 0.1}, 'seed': args.seed, 'note': 'Grouped identical normalized bodies; not label-stratified. Review rare-label support and near-duplicate versions manually.'})
    save_json(output / 'environment.json', {name: importlib.metadata.version(name) for name in ['torch', 'torch-geometric', 'transformers', 'pyvi', 'numpy', 'scikit-learn']})
    device = torch.device(args.device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    set_seed(args.seed)
    encoder = ContextEncoder(args.encoder, window=args.window, stride=args.stride, batch_size=args.encoder_batch, device=device, revision=args.revision, local_files_only=args.local_files_only)
    config = vars(args).copy()
    config['encoder_config'] = encoder.config
    config['thresholds'] = [0.2, 0.35, 0.5, 0.65, 0.8]
    config['relation_validation_setting'] = 'predicted_entities_end_to_end'
    config['feature_note'] = 'RE positions: relative start/end, clipped span unit count, clipped START unit index (not document length).'
    save_json(output / 'run_config.json', config)
    features = {}
    for i, doc in enumerate(documents):
        encoded = encoder.encode(doc['text'], output / 'cache')
        features[doc['id']] = str(output / 'cache' / f"{encoded['cache_key']}.pt")
        print(f"Embedding {i + 1}/{len(documents)}: {doc['id']} ({len(encoded['offsets'])} units)", flush=True)
    del encoded, encoder
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    ner, ner_threshold = train_ner(splits, features, config, output, device)
    ner_test = evaluate_ner(ner, splits['test'], features, device, ner_threshold)
    save_json(output / 'ner_test.json', ner_test)
    graphs = {doc['id']: graph_for(doc, read_features(features[doc['id']]), config) for doc in documents}
    inferred_graphs = {}
    for doc in splits['val'] + splits['test']:
        feature = read_features(features[doc['id']])
        inferred = predict_entities(ner, feature, doc['text'], device, ner_threshold)
        inferred_graphs[doc['id']] = graph_for(doc, feature, config, inferred)
    results = []
    for variant in args.variants.split(','):
        if variant not in {'semantic', 'rgcn'}:
            raise ValueError(f'Unknown variant: {variant}')
        relation, threshold = train_relations(splits, graphs, config, output, device, variant, validation_graphs=inferred_graphs)
        oracle = evaluate_relations(relation, splits['test'], graphs, device, threshold)
        end_to_end = evaluate_relations(relation, splits['test'], inferred_graphs, device, threshold)
        save_json(output / f'test_{variant}.json', {'gold_entities': oracle, 'predicted_entities_end_to_end': end_to_end})
        for setting, metrics in [('gold_entities', oracle), ('predicted_entities_end_to_end', end_to_end)]:
            results.append({'variant': variant, 'setting': setting, **{k: v for k, v in metrics.items() if k != 'per_class'}})
    if results:
        with (output / 'comparison.csv').open('w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(results[0]))
            writer.writeheader()
            writer.writerows(results)
    selected = {}
    for name in ['ner'] + [f'relation_{v}' for v in args.variants.split(',')]:
        saved = load_checkpoint(output / 'checkpoints' / f'{name}.pt', 'cpu')
        selected[name] = {'epoch': saved['epoch'], 'threshold': saved['threshold'], 'validation': saved['validation']}
    save_json(output / 'selected_checkpoints.json', selected)
    save_json(output / 'run_status.json', {'status': 'complete', 'version': PIPELINE_VERSION, 'elapsed_seconds': time.perf_counter() - started, 'device': str(device), 'device_name': torch.cuda.get_device_name(device) if device.type == 'cuda' else 'CPU'})
    print(f'Finished. Actual measured results: {output}', flush=True)

def parser():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--dataset', required=True)
    p.add_argument('--output', default='output_results/raw_doc_v1')
    p.add_argument('--audit-only', action='store_true')
    p.add_argument('--encoder', default='vinai/phobert-base-v2')
    p.add_argument('--revision', default='main')
    p.add_argument('--local-files-only', action='store_true')
    p.add_argument('--device', default=None)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--window', type=int, default=256)
    p.add_argument('--stride', type=int, default=128)
    p.add_argument('--encoder-batch', type=int, default=8)
    p.add_argument('--token-batch', type=int, default=2048)
    p.add_argument('--ner-epochs', type=int, default=20)
    p.add_argument('--relation-epochs', type=int, default=30)
    p.add_argument('--patience', type=int, default=5)
    p.add_argument('--hidden', type=int, default=128)
    p.add_argument('--dropout', type=float, default=0.2)
    p.add_argument('--lr', type=float, default=0.001)
    p.add_argument('--neg-ratio', type=float, default=5.0)
    p.add_argument('--neighbors', type=int, default=8)
    p.add_argument('--anchors', type=int, default=12)
    p.add_argument('--enclosing', type=int, default=2)
    p.add_argument('--selection-metric', choices=['f1_micro_positive', 'f1_macro_22'], default='f1_micro_positive')
    p.add_argument('--variants', default='semantic,rgcn')
    return p

class Extractor:

    def __init__(self, model_dir, variant='rgcn', device=None, local_files_only=False):
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        model_dir = Path(model_dir)
        self.model_dir = model_dir
        ner_ckpt = load_checkpoint(model_dir / 'checkpoints' / 'ner.pt', self.device)
        re_ckpt = load_checkpoint(model_dir / 'checkpoints' / f'relation_{variant}.pt', self.device)
        self.config = ner_ckpt['config']
        if re_ckpt['config'] != self.config or re_ckpt.get('variant') != variant:
            raise ValueError('NER and relation checkpoints must belong to the same run')
        self.ner = BoundaryNER(self.config['hidden'], self.config['dropout']).to(self.device)
        self.relation = LegalRGCN(self.config['hidden'], self.config['dropout'], variant == 'rgcn').to(self.device)
        self.ner.load_state_dict(ner_ckpt['model_state_dict'], strict=True)
        self.relation.load_state_dict(re_ckpt['model_state_dict'], strict=True)
        self.ner.eval()
        self.relation.eval()
        self.ner_threshold = ner_ckpt['threshold']
        self.relation_threshold = re_ckpt['threshold']
        ec = self.config['encoder_config']
        self.encoder = ContextEncoder(ec['model_name'], ec['window'], ec['stride'], self.config['encoder_batch'], self.device, ec['revision'], local_files_only)
        if self.encoder.config != ec:
            raise ValueError('Encoder/segmentation versions differ from training; use the recorded environment')

    @torch.no_grad()
    def predict(self, text):
        if not isinstance(text, str) or not text.strip():
            raise ValueError('Văn bản đầu vào không được rỗng.')
        if self.device.type == 'cuda':
            torch.cuda.synchronize(self.device)
        started = time.perf_counter()
        text = body_text(text)
        if not text.strip():
            raise ValueError('Không có phần thân văn bản để nhận diện.')
        feature = self.encoder.encode(text)
        entities = predict_entities(self.ner, feature, text, self.device, self.ner_threshold)
        if len(entities) > 10000:
            raise ValueError(f'NER predicted {len(entities)} entities, exceeding the 10,000-node resource guard. Inspect NER training/threshold and input quality, or test a shorter document. No predictions were silently truncated.')
        graph = graph_for({'text': text}, feature, self.config, entities)
        relations = []
        if graph['pairs']:
            h = self.relation.encode_nodes(graph['x'].to(self.device), graph['edge_index'].to(self.device), graph['edge_type'].to(self.device))
            for first in range(0, len(graph['pairs']), 2048):
                pairs = graph['pairs'][first:first + 2048]
                probabilities = self.relation.classify(h, torch.tensor(pairs, device=self.device).t()).sigmoid().cpu()
                for i, r in (probabilities >= self.relation_threshold).nonzero().tolist():
                    u, v = pairs[i]
                    relations.append({'from_id': entities[u]['id'], 'to_id': entities[v]['id'], 'code': ALL_RELATIONS[r], 'score': float(probabilities[i, r])})
        if self.device.type == 'cuda':
            torch.cuda.synchronize(self.device)
        return {'full_text': text, 'entities': entities, 'relations': relations, 'inference': {'elapsed_seconds': time.perf_counter() - started, 'characters': len(text), 'lexical_units': len(feature['offsets']), 'candidate_pairs': len(graph['pairs']), 'ner_threshold': self.ner_threshold, 'relation_threshold': self.relation_threshold, 'device': str(self.device), 'timing_scope': 'Encoding + NER + RE; excludes model loading, DOCX reading and visualization.'}, 'note': 'Scores are uncalibrated model outputs; absent candidate pairs are not scored.'}

    def predict_file(self, path):
        text, info = read_input_document(path)
        predicted = self.predict(text)
        matches = []
        manifest_path = self.model_dir / 'split_manifest.json'
        if manifest_path.exists():
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
            value = normalized_body_hash(predicted['full_text'])
            ids = {i for i, h in manifest.get('body_hashes', {}).items() if h == value}
            matches = [name for name, items in manifest['splits'].items() if ids.intersection(items)]
        info['matching_splits_by_exact_normalized_body'] = matches
        info['evaluation_note'] = 'Nội dung trùng với tập ' + ', '.join(matches) + '. Chỉ dùng minh họa; không coi là kiểm tra độc lập mới.' if matches else 'Chưa phát hiện trùng nội dung chuẩn hóa với manifest. Cần tự kiểm tra bản gần trùng; chưa có nhãn chuẩn để chấm điểm.'
        info['inference'] = predicted['inference']
        return (predicted, info)

# RUN TRAINING
if globals().get('TRAIN_NEW_MODEL', True):
    args = parser().parse_args([
        '--dataset', str(DATASET_DIR), '--output', str(OUTPUT_DIR),
        '--seed', str(SEED), '--encoder', ENCODER,
        '--ner-epochs', str(NER_EPOCHS), '--relation-epochs', str(RELATION_EPOCHS),
        '--encoder-batch', str(ENCODER_BATCH), '--variants', VARIANTS,
        '--neg-ratio', str(NEGATIVE_RATIO), '--neighbors', str(NEIGHBORS),
        '--anchors', str(ANCHORS), '--enclosing', str(ENCLOSING_CANDIDATES),
        '--lr', str(LEARNING_RATE), '--dropout', str(DROPOUT),
        '--selection-metric', 'f1_micro_positive'
    ]+(['--local-files-only'] if LOCAL_FILES_ONLY else []))
    run(args)
    print('🎉 Hoàn tất huấn luyện và đánh giá!')
else:
    print('Đã nạp kiến trúc và bộ suy luận. Không huấn luyện lại; chuyển đến Bước 11.')


### 📊 [BƯỚC 8] Vẽ quá trình huấn luyện NER và RE
Hai tác vụ được vẽ riêng: loss và F1 validation. Tất cả hình được lưu PNG để chèn vào báo cáo.


In [ ]:
"""Measured evaluation reports; separate NER/RE tasks and preserve raw metrics."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def generate_training_report(output):
    output = Path(output)

    def read(name):
        return json.loads((output / name).read_text(encoding='utf-8'))
    ner = read('ner_test.json')
    figures = output / 'figures'
    figures.mkdir(exist_ok=True)
    saved = []

    def finish(fig, name):
        fig.tight_layout()
        fig.savefig(figures / name, dpi=180, bbox_inches='tight')
        plt.close(fig)
        saved.append(str(figures / name))
    history = read('ner_history.json')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = [h['epoch'] for h in history]
    axes[0].plot(epochs, [h['loss'] for h in history], marker='.')
    axes[0].set(title='NER: training loss', xlabel='Epoch', ylabel='Weighted BCE')
    axes[1].plot(epochs, [100 * h['val_f1'] for h in history], marker='.')
    axes[1].set(title='NER: validation exact-span/type F1', xlabel='Epoch', ylabel='F1 (%)', ylim=(0, 100))
    for ax in axes:
        ax.grid(alpha=0.25)
    finish(fig, 'ner_training.png')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    rows = []
    for variant in ('semantic', 'rgcn'):
        file = output / f'test_{variant}.json'
        if not file.exists():
            continue
        h = read(f'relation_{variant}_history.json')
        x = [a['epoch'] for a in h]
        axes[0].plot(x, [a['loss'] for a in h], label=variant, marker='.')
        metric = 'val_f1_micro_positive' if 'val_f1_micro_positive' in h[0] else 'val_f1_macro_22'
        axes[1].plot(x, [100 * a[metric] for a in h], label=f'{variant}: {metric}', marker='.')
        for setting, scores in read(file.name).items():
            rows.append({'model': variant, 'setting': setting, 'precision_micro_pct': 100 * scores['precision_micro_positive'], 'recall_micro_pct': 100 * scores['recall_micro_positive'], 'f1_micro_pct': 100 * scores['f1_micro_positive'], 'f1_macro_22_pct': 100 * scores['f1_macro_22'], 'candidate_recall_pct': 100 * scores['candidate_recall'], 'supported_relations': scores['supported_classes']})
            per = pd.DataFrame(scores['per_class']).T
            per['fn'] = per['support'] - per['tp']
            per.to_csv(output / f'relations_{variant}_{setting}.csv', encoding='utf-8-sig', index_label='code')
    axes[0].set(title='RE: training loss', xlabel='Epoch', ylabel='Weighted BCE')
    axes[1].set(title='RE: validation selection score', xlabel='Epoch', ylabel='F1 (%)', ylim=(0, 100))
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(fontsize=7)
    finish(fig, 're_training.png')
    main = pd.DataFrame(rows)
    main.to_csv(output / 'report_relations.csv', index=False, encoding='utf-8-sig')
    summary = pd.DataFrame([{'task': 'NER exact-span/type', 'precision_pct': 100 * ner['precision'], 'recall_pct': 100 * ner['recall'], 'f1_pct': 100 * ner['f1'], 'tp': ner['tp'], 'fp': ner['fp'], 'fn': ner['fn']}])
    summary.to_csv(output / 'report_ner.csv', index=False, encoding='utf-8-sig')
    entities = pd.DataFrame(ner['per_type']).T
    entities['support'] = entities['tp'] + entities['fn']
    entities.to_csv(output / 'entities_per_type.csv', encoding='utf-8-sig', index_label='code')
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(['Precision', 'Recall', 'F1'], [100 * ner[k] for k in ('precision', 'recall', 'f1')], color=['#2563eb', '#0d9488', '#7c3aed'])
    ax.set(title='NER test: exact-span AND entity type', ylim=(0, 100), ylabel='%')
    finish(fig, 'ner_test.png')
    fig, ax = plt.subplots(figsize=(11, 5))
    end = main[main.setting == 'predicted_entities_end_to_end']
    x = np.arange(len(end))
    width = 0.2
    for j, col in enumerate(['precision_micro_pct', 'recall_micro_pct', 'f1_micro_pct', 'f1_macro_22_pct']):
        ax.bar(x + (j - 1.5) * width, end[col], width, label=col.replace('_pct', ''))
    ax.set_xticks(x, end.model)
    ax.set(title='RE test: predicted entities (end-to-end)', ylabel='%', ylim=(0, 100))
    ax.legend(fontsize=8)
    finish(fig, 're_test_comparison.png')
    fig, ax = plt.subplots(figsize=(12, 11))
    ax.barh(entities.index[::-1], 100 * entities.f1.iloc[::-1], color='#0d9488')
    ax.set(title='NER test F1 by entity type (zero includes unsupported types)', xlabel='F1 (%)', xlim=(0, 100))
    ax.tick_params(axis='y', labelsize=7)
    finish(fig, 'ner_per_type.png')
    preferred = 'rgcn' if (output / 'test_rgcn.json').exists() else 'semantic'
    scores = read(f'test_{preferred}.json')['predicted_entities_end_to_end']
    per = pd.DataFrame(scores['per_class']).T
    per['fn'] = per.support - per.tp
    matrix = per[['tp', 'fp', 'fn']].to_numpy(dtype=int)
    fig, ax = plt.subplots(figsize=(13, 9))
    ax.imshow(np.log1p(matrix), cmap='Blues', aspect='auto')
    ax.set_xticks(range(3), ['TP', 'FP', 'FN'])
    ax.set_yticks(range(len(per)), per.index, fontsize=7)
    for i in range(len(per)):
        for j in range(3):
            color = 'white' if np.log1p(matrix[i, j]) > np.log1p(matrix.max()) * 0.55 else '#172033'
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center', fontsize=8, color=color)
    ax.set_title(f'RE {preferred} end-to-end: counts by relation (log color scale)')
    finish(fig, 're_errors.png')
    note = '# Kết quả thực đo\n\nNER: đúng đồng thời ranh giới và loại. RE: quan hệ dương có hướng; kết quả chính dùng thực thể NER dự đoán.\n\nMacro-F1 lấy đủ 22 lớp, lớp không có mẫu có F1=0. Đọc cùng support; micro-F1 có thể bị THUOC chi phối.\n\nLoss là loss huấn luyện có trọng số, không phải phần trăm chính xác. Các biểu đồ không chứng minh khả năng khái quát nếu nhãn test chưa được người kiểm tra.\n\n'
    (output / 'REPORT.md').write_text(note + summary.round(2).to_markdown(index=False) + '\n\n' + main.round(2).to_markdown(index=False), encoding='utf-8')
    return {'ner': summary, 'relations': main, 'figures': saved}

from IPython.display import display, Image

status_path = OUTPUT_DIR/'run_status.json'
if not status_path.exists() or json.loads(status_path.read_text(encoding='utf-8')).get('status') != 'complete':
    raise RuntimeError('Chưa có lần huấn luyện v2 hoàn tất. Chạy Bước 7 hoặc chọn đúng thư mục kết quả.')
report = generate_training_report(OUTPUT_DIR)
display(Image(filename=str(OUTPUT_DIR/'figures'/'ner_training.png')))
display(Image(filename=str(OUTPUT_DIR/'figures'/'re_training.png')))
print('Đã lưu 6 biểu đồ PNG, bảng CSV và REPORT.md trong thư mục kết quả.')


### 🎯 [BƯỚC 9] Kết quả test chính
NER: Precision/Recall/F1 exact-span/type. RE: micro-Precision/Recall/F1 và Macro-F1 22 nhãn.
Hai chế độ thực thể được trình bày riêng. Không có số điểm đặt sẵn.


In [ ]:
ner_table = pd.read_csv(OUTPUT_DIR/'report_ner.csv')
display(ner_table.rename(columns={'task':'Tác vụ','precision_pct':'Precision (%)','recall_pct':'Recall (%)','f1_pct':'F1 (%)'}).round(2))
re_table = pd.read_csv(OUTPUT_DIR/'report_relations.csv')
main_columns = ['model','setting','precision_micro_pct','recall_micro_pct','f1_micro_pct','f1_macro_22_pct']
display(re_table[main_columns].rename(columns={'model':'Mô hình','setting':'Chế độ','precision_micro_pct':'Precision micro (%)','recall_micro_pct':'Recall micro (%)','f1_micro_pct':'F1 micro (%)','f1_macro_22_pct':'Macro-F1 22 (%)'}).round(2))
display(Image(filename=str(OUTPUT_DIR/'figures'/'ner_test.png')))
display(Image(filename=str(OUTPUT_DIR/'figures'/'re_test_comparison.png')))
print('Kết quả chính: RE predicted_entities_end_to_end. Gold_entities chỉ đo riêng RE khi đã biết thực thể.')
print('Macro-F1 tính đủ 22 nhãn. Candidate recall và số lớp có mẫu nằm trong report_relations.csv; chi tiết theo nhãn ở Bước 10.')


### 📋 [BƯỚC 10] Phân tích nhãn và lỗi
Xem F1 theo loại thực thể và TP/FP/FN theo quan hệ. Chi tiết số liệu nằm trong CSV; mặc định không in quá nhiều bảng.


In [ ]:
display(Image(filename=str(OUTPUT_DIR/'figures'/'ner_per_type.png')))
display(Image(filename=str(OUTPUT_DIR/'figures'/'re_errors.png')))
SHOW_DETAILED_TABLES = False
if SHOW_DETAILED_TABLES:
    display(pd.read_csv(OUTPUT_DIR/'entities_per_type.csv').round(3))
    for variant in ['semantic','rgcn']:
        path = OUTPUT_DIR/f'relations_{variant}_predicted_entities_end_to_end.csv'
        if path.exists(): display(pd.read_csv(path).round(3))
print('Bảng chính: report_ner.csv, report_relations.csv. Chi tiết: entities_per_type.csv, relations_*.csv.')
print('Để đánh giá thực tế: đối chiếu một bộ văn bản mới được người kiểm tra gán nhãn; không chỉ chọn vài kết quả đúng.')


### 🔍 [BƯỚC 11] Tải DOCX và xem đồ thị quan hệ dự đoán
Tải lên một DOCX (hoặc TXT/JSON có full_text). Đọc đoạn văn và bảng theo thứ tự, không cần cung cấp nhãn.
Xuất văn bản có đánh dấu thực thể, bảng kết quả và đồ thị HTML tương tác có bộ lọc, kéo/phóng to, chọn nút để xem nguồn.
PNG là bản xem gọn; JSON/CSV giữ đầy đủ kết quả. Không có nhãn chuẩn thì không tự tính Precision/Recall/F1.


In [ ]:
"""Read DOCX/TXT without changing text after offsets have been predicted."""
import json
import hashlib
from pathlib import Path
from zipfile import ZipFile, BadZipFile
from xml.etree import ElementTree as ET
W_NS = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'

def _paragraph_text(element):
    parts = []

    def visit(node):
        if node.tag in {W_NS + 'del', W_NS + 'instrText'}:
            return
        if node.tag == W_NS + 't':
            parts.append(node.text or '')
        elif node.tag == W_NS + 'tab':
            parts.append('\t')
        elif node.tag in {W_NS + 'br', W_NS + 'cr'}:
            parts.append('\n')
        else:
            for child in node:
                visit(child)
    visit(element)
    return ''.join(parts)

def _docx_blocks(parent):
    blocks = []
    for child in parent:
        if child.tag == W_NS + 'p':
            blocks.append(_paragraph_text(child))
        elif child.tag == W_NS + 'tbl':
            for row in child.findall(W_NS + 'tr'):
                cells = ['\n'.join(_docx_blocks(cell)) for cell in row.findall(W_NS + 'tc')]
                blocks.append('\t'.join(cells))
        elif child.tag in {W_NS + 'sdt', W_NS + 'sdtContent', W_NS + 'ins'}:
            blocks.extend(_docx_blocks(child))
    return blocks

def read_input_document(path):
    """Body paragraphs/tables in document order; scanned images require OCR separately."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy tệp: {path}')
    suffix = path.suffix.lower()
    if suffix == '.docx':
        try:
            with ZipFile(path) as archive:
                root = ET.fromstring(archive.read('word/document.xml'))
                body = root.find(W_NS + 'body')
                text = '\n'.join(_docx_blocks(body)) if body is not None else ''
        except (BadZipFile, KeyError, ET.ParseError) as exc:
            raise ValueError(f'DOCX không hợp lệ hoặc có mật khẩu: {path.name}') from exc
    elif suffix == '.txt':
        text = path.read_text(encoding='utf-8-sig')
    elif suffix == '.json':
        raw = json.loads(path.read_text(encoding='utf-8-sig'))
        if isinstance(raw, list) and len(raw) == 1:
            raw = raw[0]
        text = raw.get('full_text') if isinstance(raw, dict) else None
        if not isinstance(text, str):
            raise ValueError('JSON phải có full_text; nhãn cung cấp không được dùng khi suy luận.')
    else:
        raise ValueError('Chỉ hỗ trợ .docx, .txt hoặc JSON có full_text; .doc/PDF cần chuyển đổi trước.')
    if not text.strip():
        raise ValueError('Không đọc được văn bản. Nếu tệp chỉ chứa ảnh quét, cần OCR trước.')
    return (text, {'name': path.name, 'format': suffix, 'file_sha256': hashlib.sha256(path.read_bytes()).hexdigest(), 'text_sha256': hashlib.sha256(text.encode('utf-8')).hexdigest(), 'characters': len(text), 'extraction_note': 'DOCX: thân văn bản và bảng theo thứ tự; không lấy header/footer, chú thích hoặc OCR ảnh. Offset tính trên input_text.txt, không phải số trang Word.'})

"""Export real predictions to CSV, annotated text and a standalone interactive SVG graph."""
import html
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

def _marked_source(text, entities):
    events = {}
    for e in entities:
        events.setdefault(e['start'], []).append((1, e['id']))
        events.setdefault(e['end'], []).append((-1, e['id']))
    by_id = {e['id']: e for e in entities}
    points = sorted({0, len(text), *events})
    active = set()
    parts = []
    for a, b in zip(points, points[1:]):
        for action, eid in events.get(a, []):
            if action == 1:
                active.add(eid)
            else:
                active.discard(eid)
        value = html.escape(text[a:b])
        title = html.escape(' | '.join((f"{i}: {by_id[i]['code']}" for i in sorted(active))))
        tag = 'mark' if active else 'span'
        parts.append(f'<{tag} id="offset-{a}" title="{title}">{value}</{tag}>')
    return ''.join(parts)

def prediction_dashboard(prediction, title='Kết quả nhận diện', source_info=None):
    entities = prediction['entities']
    relations = prediction['relations']
    view_relations = sorted(relations, key=lambda r: -r['score'])[:20000]
    payload = {'entities': entities, 'relations': view_relations, 'relation_names': REL_VN_NAMES, 'entity_names': {k: v[1] for k, v in ENTITY_TAXONOMY.items()}, 'total_relations': len(relations)}
    data = json.dumps(payload, ensure_ascii=False).replace('<', '\\u003c').replace('>', '\\u003e').replace('&', '\\u0026')
    source_note = (source_info or {}).get('evaluation_note', 'Đây là dự đoán chưa được đối chiếu với nhãn chuẩn.')
    template = '<!doctype html><html lang="vi"><meta charset="utf-8"><title>__TITLE__</title>\n<style>body{font:15px Arial;margin:20px;color:#172033;background:#f5f7fb}h1{font-size:23px}.panel{background:white;padding:16px;border:1px solid #ddd;border-radius:8px;margin:12px 0}button,select,input{padding:7px;margin:4px}#graph{width:100%;height:650px;background:#fff;border:1px solid #ccc;touch-action:none}#source{white-space:pre-wrap;line-height:1.7;max-height:520px;overflow:auto}mark{background:#fff1a6}mark:target{background:#fb923c}.small{color:#475569;font-size:13px}.node{cursor:grab}.edge-label{font-size:11px;paint-order:stroke;stroke:white;stroke-width:4px;stroke-linejoin:round;fill:#334155}.node text{font-size:12px;pointer-events:none}#detail{white-space:pre-wrap;max-height:300px;overflow:auto}a{color:#1d4ed8}</style>\n<h1>__TITLE__</h1><div class="panel">__NOTE__<br><span class="small">Điểm mô hình chưa được hiệu chỉnh thành xác suất đúng. Cạnh trong hình là quan hệ dự đoán, không phải cạnh vị trí đầu vào RGCN. Không có nhãn chuẩn thì không tính được Precision/Recall/F1.</span></div>\n<div class="panel"><label>Quan hệ <select id="relation"><option value="">Tất cả</option></select></label>\n<label>Điểm tối thiểu <input id="score" type="number" min="0" max="1" step=".05" value="0"></label>\n<label>Số nút tối đa <input id="nodes" type="number" min="2" max="200" value="45"></label>\n<label>Số cạnh tối đa <input id="edges" type="number" min="1" max="500" value="80"></label>\n<button id="draw">Áp dụng</button><button id="reset">Đặt lại góc nhìn</button><button id="svgSave">Tải SVG</button>\n<div id="counts" class="small"></div><div class="small">Cuộn để phóng to; kéo nền để di chuyển; kéo nút để sắp xếp; bấm nút để xem văn bản và các quan hệ.</div>\n<svg id="graph" viewBox="0 0 1200 650"><defs><marker id="arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0 0L10 5L0 10Z" fill="#64748b"/></marker></defs><g id="scene"></g></svg></div>\n<div class="panel"><b>Chi tiết thực thể và quan hệ</b><div id="detail">Chọn một nút trên đồ thị.</div></div>\n<div class="panel"><b>Văn bản nguồn và các thực thể được đánh dấu</b><div class="small">Di chuột lên vùng tô màu để xem nhãn. Thực thể chồng lấn có thể cùng xuất hiện trong một vùng.</div><div id="source">__SOURCE__</div></div>\n<script type="application/json" id="data">__DATA__</script><script>\nconst D=JSON.parse(document.getElementById(\'data\').textContent), byId=new Map(D.entities.map(e=>[e.id,e]));\nconst svg=document.getElementById(\'graph\'),scene=document.getElementById(\'scene\'),NS=\'http://www.w3.org/2000/svg\';\nlet view=[0,0,1200,650],points=[],links=[],moving=null,dragged=false;\nfunction el(tag,attrs,parent){const n=document.createElementNS(NS,tag);for(const [k,v] of Object.entries(attrs||{}))n.setAttribute(k,v);if(parent)parent.appendChild(n);return n;}\nfunction label(code){return D.relation_names[code]||code;}\nfunction updateView(){svg.setAttribute(\'viewBox\',view.join(\' \'));}\nfunction details(e){const box=document.getElementById(\'detail\');box.textContent=e.id+\' — \'+(D.entity_names[e.code]||e.code)+\'\\nVị trí [\'+e.start+\', \'+e.end+\')\\nĐiểm NER: \'+e.score.toFixed(3)+\'\\n\\n\'+e.text+\'\\n\\n\';\n const related=D.relations.filter(r=>r.from_id===e.id||r.to_id===e.id);box.appendChild(document.createTextNode(\'Hiển thị \'+Math.min(100,related.length)+\'/\'+related.length+\' quan hệ trong HTML; CSV giữ đầy đủ.\\n\'));for(const r of related.slice(0,100)){box.appendChild(document.createTextNode(r.from_id+\' → \'+label(r.code)+\' → \'+r.to_id+\' (\'+r.score.toFixed(3)+\')\\n\'));}\n const target=document.getElementById(\'offset-\'+e.start);if(target){document.querySelectorAll(\'mark\').forEach(n=>n.style.outline=\'\');target.style.outline=\'3px solid #f97316\';target.scrollIntoView({block:\'center\',behavior:\'smooth\'});}}\nfunction border(p,dx,dy){const k=Math.min(85/Math.max(.001,Math.abs(dx)),27/Math.max(.001,Math.abs(dy)));return {x:p.x+dx*k,y:p.y+dy*k};}\nfunction paths(){for(const l of links){const a=l.a,b=l.b,dx=b.x-a.x,dy=b.y-a.y;let bend=l.parallel?35:0;const cx=(a.x+b.x)/2-dy*.12*bend/35,cy=(a.y+b.y)/2+dx*.12*bend/35;const start=border(a,cx-a.x,cy-a.y),end=border(b,cx-b.x,cy-b.y);l.path.setAttribute(\'d\',`M${start.x},${start.y} Q${cx},${cy} ${end.x},${end.y}`);l.text.setAttribute(\'x\',cx);l.text.setAttribute(\'y\',cy-6);}for(const p of points)p.group.setAttribute(\'transform\',`translate(${p.x},${p.y})`);}\nfunction draw(){scene.replaceChildren();const type=document.getElementById(\'relation\').value,score=Number(document.getElementById(\'score\').value)||0;\n const maxN=Math.max(2,Math.min(200,Number(document.getElementById(\'nodes\').value)||45)),maxE=Math.max(1,Math.min(500,Number(document.getElementById(\'edges\').value)||80));\n const eligible=D.relations.filter(r=>(!type||r.code===type)&&r.score>=score&&byId.has(r.from_id)&&byId.has(r.to_id));const ids=new Set(),chosen=[],groups=new Map();\n for(const r of eligible){const key=r.from_id+\'|\'+r.to_id;if(!groups.has(key))groups.set(key,{...r,codes:[]});groups.get(key).codes.push(r.code);}\n for(const r of groups.values()){const extra=Number(!ids.has(r.from_id))+Number(!ids.has(r.to_id));if(ids.size+extra>maxN)continue;ids.add(r.from_id);ids.add(r.to_id);chosen.push(r);if(chosen.length>=maxE)break;}\n if(!ids.size)for(const e of D.entities.slice(0,maxN))ids.add(e.id);\n const degree=new Map([...ids].map(id=>[id,0]));for(const r of chosen){degree.set(r.from_id,degree.get(r.from_id)+1);degree.set(r.to_id,degree.get(r.to_id)+1);}\n points=[...ids].map((id,i)=>({id,x:600+240*Math.cos(i*2*Math.PI/ids.size),y:320+210*Math.sin(i*2*Math.PI/ids.size)}));const map=new Map(points.map(p=>[p.id,p]));\n for(let t=0;t<150;t++){const f=new Map(points.map(p=>[p.id,{x:0,y:0}]));for(let i=0;i<points.length;i++)for(let j=i+1;j<points.length;j++){const a=points[i],b=points[j],dx=a.x-b.x,dy=a.y-b.y,d=Math.max(20,Math.hypot(dx,dy)),k=1600/(d*d);f.get(a.id).x+=dx*k;f.get(a.id).y+=dy*k;f.get(b.id).x-=dx*k;f.get(b.id).y-=dy*k;}\n for(const r of chosen){const a=map.get(r.from_id),b=map.get(r.to_id),dx=b.x-a.x,dy=b.y-a.y,d=Math.max(1,Math.hypot(dx,dy)),k=(d-145)*.003;f.get(a.id).x+=dx*k;f.get(a.id).y+=dy*k;f.get(b.id).x-=dx*k;f.get(b.id).y-=dy*k;}\n for(const p of points){p.x=Math.max(100,Math.min(1100,p.x+Math.max(-5,Math.min(5,f.get(p.id).x))));p.y=Math.max(65,Math.min(585,p.y+Math.max(-5,Math.min(5,f.get(p.id).y))));}}\n links=chosen.map(r=>{const path=el(\'path\',{fill:\'none\',stroke:r.codes.length===1&&r.code===\'THUOC\'?\'#94a3b8\':\'#2563eb\',\'stroke-width\':1.6,\'marker-end\':\'url(#arrow)\'},scene),txt=el(\'text\',{\'class\':\'edge-label\',\'text-anchor\':\'middle\'},scene);txt.textContent=label(r.code).slice(0,22)+(r.codes.length>1?\' +\'+(r.codes.length-1):\'\');const title=el(\'title\',{},path);title.textContent=r.codes.map(label).join(\'\\n\')+\' / max score \'+r.score.toFixed(3);return {a:map.get(r.from_id),b:map.get(r.to_id),path,text:txt,parallel:chosen.some(s=>s.from_id===r.to_id&&s.to_id===r.from_id)};});\n for(const p of points){const e=byId.get(p.id),g=el(\'g\',{\'class\':\'node\'},scene);p.group=g;el(\'rect\',{x:-82,y:-24,width:164,height:48,rx:8,fill:e.code===\'DIEU\'||e.code===\'KHOAN\'||e.code===\'DIEM\'?\'#dbeafe\':\'#dcfce7\',stroke:\'#475569\'},g);const a=el(\'text\',{\'text-anchor\':\'middle\',y:-4},g);a.textContent=p.id+\' · \'+(D.entity_names[e.code]||e.code).slice(0,19);const b=el(\'text\',{\'text-anchor\':\'middle\',y:14},g);b.textContent=e.text.replace(/\\s+/g,\' \').slice(0,23);el(\'title\',{},g).textContent=e.code+\'\\n\'+e.text;\n g.addEventListener(\'pointerdown\',ev=>{ev.stopPropagation();moving={point:p,x:ev.clientX,y:ev.clientY};dragged=false;svg.setPointerCapture(ev.pointerId);});g.addEventListener(\'click\',()=>{if(!dragged)details(e);});}\n paths();document.getElementById(\'counts\').textContent=`Toàn bộ: ${D.entities.length} thực thể, ${D.total_relations} quan hệ. Khớp bộ lọc trong HTML: ${eligible.length}. Đang vẽ ${points.length} nút / ${links.length} cặp có hướng (gộp nhiều nhãn/cặp). JSON/CSV giữ đầy đủ kết quả; HTML xét tối đa 20.000 quan hệ có điểm cao nhất.`;}\nfor(const code of [...new Set(D.relations.map(r=>r.code))].sort()){const o=document.createElement(\'option\');o.value=code;o.textContent=label(code);document.getElementById(\'relation\').appendChild(o);}\ndocument.getElementById(\'draw\').onclick=draw;document.getElementById(\'reset\').onclick=()=>{view=[0,0,1200,650];updateView();};\nsvg.addEventListener(\'wheel\',e=>{e.preventDefault();const k=e.deltaY>0?1.12:1/1.12;view=[view[0]+view[2]*(1-k)/2,view[1]+view[3]*(1-k)/2,view[2]*k,view[3]*k];updateView();},{passive:false});\nsvg.addEventListener(\'pointerdown\',e=>{moving={x:e.clientX,y:e.clientY};svg.setPointerCapture(e.pointerId);});\nsvg.addEventListener(\'pointermove\',e=>{if(!moving)return;const dx=(e.clientX-moving.x)*view[2]/svg.clientWidth,dy=(e.clientY-moving.y)*view[3]/svg.clientHeight;if(Math.abs(dx)+Math.abs(dy)>1)dragged=true;if(moving.point){moving.point.x+=dx;moving.point.y+=dy;paths();}else{view[0]-=dx;view[1]-=dy;updateView();}moving.x=e.clientX;moving.y=e.clientY;});\nsvg.addEventListener(\'pointerup\',()=>{if(moving&&moving.point&&!dragged)details(byId.get(moving.point.id));moving=null;});svg.addEventListener(\'pointercancel\',()=>moving=null);\ndocument.getElementById(\'svgSave\').onclick=()=>{const blob=new Blob([new XMLSerializer().serializeToString(svg)],{type:\'image/svg+xml\'}),url=URL.createObjectURL(blob),a=document.createElement(\'a\');a.href=url;a.download=\'relation_graph.svg\';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000);};draw();\n</script></html>'
    return template.replace('__TITLE__', html.escape(title)).replace('__NOTE__', html.escape(source_note)).replace('__SOURCE__', _marked_source(prediction['full_text'], entities)).replace('__DATA__', data)

def export_prediction(prediction, output, source_info=None):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    save_json(output / 'prediction.json', prediction)
    save_json(output / 'source_info.json', source_info or {})
    (output / 'input_text.txt').write_text(prediction['full_text'], encoding='utf-8')
    pd.DataFrame(prediction['entities'], columns=['id', 'code', 'start', 'end', 'text', 'score']).to_csv(output / 'entities.csv', index=False, encoding='utf-8-sig')
    by_id = {e['id']: e for e in prediction['entities']}
    relations = [{**r, 'relation_name': REL_VN_NAMES.get(r['code'], r['code']), 'from_text': by_id[r['from_id']]['text'], 'to_text': by_id[r['to_id']]['text']} for r in prediction['relations']]
    pd.DataFrame(relations, columns=['from_id', 'to_id', 'code', 'relation_name', 'score', 'from_text', 'to_text']).to_csv(output / 'relations.csv', index=False, encoding='utf-8-sig')
    path = output / 'prediction.html'
    path.write_text(prediction_dashboard(prediction, source_info=source_info), encoding='utf-8')
    graph = nx.DiGraph()
    for r in sorted(prediction['relations'], key=lambda r: -r['score']):
        if len(set(graph.nodes) | {r['from_id'], r['to_id']}) > 25:
            continue
        graph.add_edge(r['from_id'], r['to_id'], label=REL_VN_NAMES.get(r['code'], r['code']))
        if graph.number_of_edges() >= 35:
            break
    if not graph:
        graph.add_nodes_from([e['id'] for e in prediction['entities'][:25]])
    fig, ax = plt.subplots(figsize=(15, 10))
    if graph:
        pos = nx.spring_layout(graph, seed=42, k=1.5)
        labels = {i: f"{i}: {by_id[i]['code']}\n{by_id[i]['text'][:25]}" for i in graph}
        nx.draw_networkx(graph, pos, labels=labels, node_color='#dbeafe', node_size=1800, font_size=7, arrows=True, ax=ax)
        nx.draw_networkx_edge_labels(graph, pos, edge_labels=nx.get_edge_attributes(graph, 'label'), font_size=6, ax=ax)
    else:
        ax.text(0.5, 0.5, 'Không có thực thể/quan hệ dự đoán', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Quan hệ dự đoán — bản xem gọn; xem HTML/CSV để kiểm tra đầy đủ')
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(output / 'relation_graph.png', dpi=180)
    plt.close(fig)
    return path

# Chọn một file DOCX mới. Trên Colab, để None để hiện nút tải tệp từ máy.
INPUT_FILE = None  # Hoặc '/content/drive/MyDrive/van_ban_moi.docx'
PREDICT_VARIANT = 'rgcn'

from datetime import datetime
from IPython.display import HTML, display

if INPUT_FILE is None:
    if IS_COLAB:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Hãy chọn đúng một file .docx hoặc .txt để kiểm tra.')
        INPUT_FILE = next(iter(uploaded))
    else:
        raise ValueError('Đặt INPUT_FILE thành đường dẫn DOCX/TXT trên máy.')

if not (OUTPUT_DIR/'checkpoints'/f'relation_{PREDICT_VARIANT}.pt').is_file():
    raise RuntimeError('Chưa có checkpoint. Chạy train hoặc chọn OUTPUT_DIR của lần chạy v2 đã hoàn tất.')
extractor = Extractor(OUTPUT_DIR, variant=PREDICT_VARIANT, local_files_only=LOCAL_FILES_ONLY)
prediction, source_info = extractor.predict_file(INPUT_FILE)
prediction_dir = OUTPUT_DIR/'predictions'/(Path(INPUT_FILE).stem+'_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
html_path = export_prediction(prediction, prediction_dir, source_info)
print(source_info['evaluation_note'])
print(f"Nhận diện: {len(prediction['entities'])} thực thể; {len(prediction['relations'])} quan hệ dự đoán.")
print('Thời gian suy luận (không gồm đọc DOCX/nạp model/vẽ):', round(prediction['inference']['elapsed_seconds'],2), 'giây')
print('Đầy đủ JSON/CSV, văn bản đã đọc, đồ thị PNG và HTML:', prediction_dir)
display(pd.read_csv(prediction_dir/'entities.csv').head(20))
display(pd.read_csv(prediction_dir/'relations.csv').head(20))
if IS_COLAB:
    # Trusted, generated local HTML; iframe keeps scripts separate from notebook UI.
    import html as html_module
    display(HTML('<iframe style="width:100%;height:1150px;border:1px solid #ddd" sandbox="allow-scripts allow-downloads" srcdoc="'+html_module.escape(html_path.read_text(encoding='utf-8'),quote=True)+'"></iframe>'))
else:
    display(HTML('<p>Mở tệp HTML bằng trình duyệt: '+str(html_path)+'</p>'))


### 📥 [TÙY CHỌN] Đóng gói & Tải Kết quả
ZIP gồm checkpoint, báo cáo và hình ảnh; bỏ cache embedding lớn. Có thể chạy lại sau Bước 11
để thêm kết quả dự đoán. File ZIP được lưu trên Drive; chỉ tải về máy khi `DOWNLOAD_ZIP = True`.


In [ ]:
import zipfile

DOWNLOAD_ZIP = False
ZIP_PATH = OUTPUT_DIR.parent/(OUTPUT_DIR.name+'_results.zip')
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        relative = path.relative_to(OUTPUT_DIR)
        if path.is_file() and relative.parts[0] != 'cache':
            archive.write(path,relative.as_posix())
print('✅ ZIP:', ZIP_PATH)
if DOWNLOAD_ZIP and IS_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))
